In [ ]:
#Cell 1
!pip install --quiet yfinance prophet statsmodels scikit-learn torch torchvision newsapi-python textblob
!pip install --quiet transformers beautifulsoup4 scipy
!pip install --quiet transformers beautifulsoup4 requests
!python -m textblob.download_corpora
!pip install --quiet openai

from textblob import TextBlob  # still available as a fallback

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.


In [ ]:
# Cell 2 — Securely capture API keys for NewsAPI and OpenAI

import os, getpass

def _set_key(env_name: str, label: str):
    """Prompt once (masked) and set as a process env var for this session only."""
    if os.environ.get(env_name):
        print(f"✅ {label} already configured")
        return
    try:
        val = getpass.getpass(f"🔑 Enter your {label}: ").strip()
        if val:
            os.environ[env_name] = val
            print(f"✅ {label} set successfully")
        else:
            print(f"⚠️ No {label} provided")
    except KeyboardInterrupt:
        print(f"⚠️ Input cancelled for {label}")

# Ask for NewsAPI (used by sentiment/news)
_set_key("NEWS_API_KEY", "NewsAPI key")

# Ask for OpenAI (used by GPT-4.1 LLM explainers)
_set_key("OPENAI_API_KEY", "OpenAI API key (for GPT-4.1)")

# Optional: lightweight verification that the OpenAI key is usable.
# If no internet or key is invalid, this will just print a friendly note.
try:
    from openai import OpenAI
    if os.environ.get("OPENAI_API_KEY"):
        client = OpenAI()  # uses OPENAI_API_KEY from env
        # A benign call that lists models; keeps output minimal and never prints your key
        _ = client.models.list()
        print("🧠 OpenAI connectivity check: OK")
    else:
        print("ℹ️ OpenAI key not provided — LLM explainers will be disabled.")
except Exception as e:
    print(f"ℹ️ Skipping OpenAI verification ({type(e).__name__}: {e})")


🔑 Enter your NewsAPI key: ··········
✅ NewsAPI key set successfully
🔑 Enter your OpenAI API key (for GPT-4.1): ··········
✅ OpenAI API key (for GPT-4.1) set successfully
🧠 OpenAI connectivity check: OK


In [ ]:

# 📦 MCP-Tradi-Win - Cell 3: Enhanced Context Schema + AgentBase + ForecastAgent + Advanced SentimentAgent
# This cell defines the shared protocol context and initializes the agent-based architecture.

# -----------------------------
# ✅ Enhanced Shared Context Schema (protocol-wide)
# -----------------------------
context = {
    "data": {
        "symbol": "BTC",
        "raw_prices": None,
        "timestamp": None
    },
    "forecast": {
        "model_used": None,
        "predicted_movement": None,  # "up", "down", or "stable"
        "confidence": None,
        "metrics_per_model": {}
    },
    "sentiment": {
        "enabled": False,
        "source": None,
        "score": None,
        "verdict": None,  # "bullish", "bearish", or "neutral"
        "confidence": None,
        # ✅ NEW ADVANCED METRICS:
        "strength_category": None,      # "Weak", "Moderate", "Strong"
        "news_volume": None,           # Number of articles found
        "recency_score": None,         # How fresh the news is (0-1)
        "directional_accuracy": None,  # Historical hit rate for sentiment
        "used_in_decision": False,
        "alignment": None,
        "headlines_sample": []         # Sample headlines for transparency
    },
    "decision": {
        "action": None,
        "rationale": None
    },
    "history": [],
    "sentiment_history": [],  # ✅ NEW: Track sentiment predictions vs actual outcomes
    "explainability": {
        "enabled": True,
        "explanation_type": "template",
        "full_trace": []
    }
}

# -----------------------------
# ✅ Simplified MCP Design - Using OrchestratorChain (Cell 6)
# -----------------------------
# Note: Complex orchestrator removed in favor of practical OrchestratorChain in Cell 6

# -----------------------------
# ✅ Agent Interface
# -----------------------------
class AgentBase:
    def run(self, context):
        raise NotImplementedError("Each agent must implement the run(context) method.")

# -----------------------------
# 🔮 Enhanced ForecastAgent: ARIMA, Prophet, LSTM with Multi-Metric Selection
# -----------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

class ForecastAgent(AgentBase):
    def __init__(self):
        # Session-based model caching
        self._cached_models = {}
        self._cache_timestamp = None

    def calculate_directional_accuracy(self, y_true, y_pred):
        """Calculate how often we predict direction correctly"""
        if len(y_true) < 2 or len(y_pred) < 2:
            return 0.0

        actual_direction = np.sign(np.diff(y_true))
        predicted_direction = np.sign(np.diff(y_pred))

        # Handle case where lengths don't match
        min_len = min(len(actual_direction), len(predicted_direction))
        if min_len == 0:
            return 0.0

        matches = (actual_direction[:min_len] == predicted_direction[:min_len]).sum()
        return float(matches / min_len)

    def composite_score(self, metrics, latest_price):
        """Combine Directional Accuracy (40%), MAPE (30%), and MAE (30%) for model selection"""
        try:
            # Normalize MAE by price to make it scale-independent
            normalized_mae = metrics["MAE"] / max(latest_price, 1e-8)

            # Convert to "goodness" scores (higher = better)
            dir_acc_score = metrics["Directional_Accuracy"]  # Already 0-1, higher is better
            mape_score = max(0, 1 - metrics["MAPE"] / 100)  # Convert MAPE to goodness score
            mae_score = max(0, 1 - normalized_mae)  # Convert normalized MAE to goodness score

            # Weighted combination: 40% Dir Acc, 30% MAPE, 30% MAE
            composite = 0.4 * dir_acc_score + 0.3 * mape_score + 0.3 * mae_score
            return composite
        except:
            return 0.0

    def run(self, context):
        symbol = context["data"]["symbol"]

        # Enhanced data fetching: 120 days instead of 90
        df = yf.download(tickers=symbol + "-USD", period="120d", interval="1d", progress=False)

        if df.empty or "Close" not in df:
            context["forecast"]["model_used"] = "none"
            context["forecast"]["predicted_movement"] = "error"
            context["forecast"]["confidence"] = 0.0
            context["explainability"]["full_trace"].append(
                "[ForecastAgent] Data fetch failed. No forecast generated."
            )
            return context

        # ✅ Assign the correct date
        context["data"]["timestamp"] = df.index[-1].strftime("%Y-%m-%d")
        print("✅ Latest Fetched Date:", context["data"]["timestamp"])

        df['ds'] = df.index
        df['y'] = df['Close']

        # Enhanced data window: 45 days instead of 30 for better patterns
        prices = df['Close'].values[-45:]
        latest_price = float(prices[-1])

        results = {}
        metrics_per_model = {}

        # === ARIMA ===
        try:
            arima_model = ARIMA(prices, order=(3,1,0)).fit()
            arima_forecast = float(arima_model.forecast()[0])
            arima_pred = arima_model.predict()

            # Use more data points for validation (last 10 instead of 5)
            validation_size = min(10, len(arima_pred))
            arima_rmse = float(np.sqrt(mean_squared_error(prices[-validation_size:], arima_pred[-validation_size:])))
            arima_mae = float(mean_absolute_error(prices[-validation_size:], arima_pred[-validation_size:]))

            # ✅ MAPE with epsilon to avoid division by zero / near-zero
            eps = 1e-8
            den = np.maximum(np.abs(prices[-validation_size:]), eps)
            arima_mape = float(np.mean(np.abs((prices[-validation_size:] - arima_pred[-validation_size:]) / den)) * 100)

            # ✅ Directional Accuracy
            arima_directional = self.calculate_directional_accuracy(prices[-validation_size:], arima_pred[-validation_size:])

            results["ARIMA"] = arima_forecast
            metrics_per_model["ARIMA"] = {
                "RMSE": arima_rmse,
                "MAE": arima_mae,
                "MAPE": arima_mape,
                "Directional_Accuracy": arima_directional
            }
        except Exception as e:
            context["explainability"]["full_trace"].append(f"[ForecastAgent] ARIMA failed: {e}")

        # === Prophet ===
        try:
            prophet = Prophet()
            prophet.fit(df[['ds', 'y']])
            future = prophet.make_future_dataframe(periods=1)
            forecast = prophet.predict(future)
            prophet_forecast = float(forecast['yhat'].iloc[-1])

            validation_size = min(10, len(forecast) - 1)
            y_true = np.array(df['y'][-validation_size:], dtype=float)
            y_pred = np.array(forecast['yhat'][-(validation_size+1):-1], dtype=float)

            prophet_rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
            prophet_mae = float(mean_absolute_error(y_true, y_pred))

            # ✅ MAPE with epsilon
            eps = 1e-8
            den = np.maximum(np.abs(y_true), eps)
            prophet_mape = float(np.mean(np.abs((y_true - y_pred) / den)) * 100)

            # ✅ Directional Accuracy
            prophet_directional = self.calculate_directional_accuracy(y_true, y_pred)

            results["Prophet"] = prophet_forecast
            metrics_per_model["Prophet"] = {
                "RMSE": prophet_rmse,
                "MAE": prophet_mae,
                "MAPE": prophet_mape,
                "Directional_Accuracy": prophet_directional
            }
        except Exception as e:
            context["explainability"]["full_trace"].append(f"[ForecastAgent] Prophet failed: {e}")

        # === Enhanced LSTM (Returns-based, Option B) ===
        try:
            # Convert prices to percentage returns
            returns = np.diff(prices) / prices[:-1] * 100  # Percentage returns

            # Scale returns for LSTM
            scaler = MinMaxScaler(feature_range=(-1, 1))
            scaled_returns = scaler.fit_transform(returns.reshape(-1, 1)).flatten()

            # Create sequences with longer lookback (12 days instead of 5)
            lookback = 12
            X, y = [], []
            for i in range(len(scaled_returns) - lookback):
                X.append(scaled_returns[i:i+lookback])
                y.append(scaled_returns[i+lookback])
            X, y = np.array(X), np.array(y)

            if len(X) > 0:
                # Enhanced LSTM architecture with dropout
                model = Sequential()
                model.add(LSTM(50, return_sequences=True, input_shape=(lookback, 1)))
                model.add(Dropout(0.2))
                model.add(LSTM(25, return_sequences=False))
                model.add(Dropout(0.2))
                model.add(Dense(1))
                model.compile(optimizer='adam', loss='mse')

                # Train with validation split
                X_reshaped = X.reshape(X.shape[0], X.shape[1], 1)
                model.fit(X_reshaped, y, epochs=20, verbose=0, validation_split=0.2)

                # Predict next return
                last_sequence = scaled_returns[-lookback:].reshape(1, lookback, 1)
                predicted_return_scaled = model.predict(last_sequence, verbose=0)[0][0]
                predicted_return = scaler.inverse_transform([[predicted_return_scaled]])[0][0]

                # Convert predicted return back to price
                lstm_forecast = float(latest_price * (1 + predicted_return / 100))

                # Calculate metrics on validation set
                val_size = min(10, len(X))
                y_pred_scaled = model.predict(X_reshaped[-val_size:], verbose=0).flatten()
                y_pred_returns = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
                y_true_returns = scaler.inverse_transform(y[-val_size:].reshape(-1, 1)).flatten()

                # Convert back to prices for metrics calculation
                base_prices = prices[-(val_size+1):-1]
                y_pred_prices = base_prices * (1 + y_pred_returns / 100)
                y_true_prices = base_prices * (1 + y_true_returns / 100)

                lstm_rmse = float(np.sqrt(mean_squared_error(y_true_prices, y_pred_prices)))
                lstm_mae = float(mean_absolute_error(y_true_prices, y_pred_prices))

                # ✅ MAPE with epsilon
                eps = 1e-8
                den = np.maximum(np.abs(y_true_prices), eps)
                lstm_mape = float(np.mean(np.abs((y_true_prices - y_pred_prices) / den)) * 100)

                # ✅ Directional Accuracy
                lstm_directional = self.calculate_directional_accuracy(y_true_prices, y_pred_prices)

                results["LSTM"] = lstm_forecast
                metrics_per_model["LSTM"] = {
                    "RMSE": lstm_rmse,
                    "MAE": lstm_mae,
                    "MAPE": lstm_mape,
                    "Directional_Accuracy": lstm_directional
                }
            else:
                raise ValueError("Insufficient data for LSTM training")

        except Exception as e:
            context["explainability"]["full_trace"].append(f"[ForecastAgent] LSTM failed: {e}")

        # === Enhanced Model Selection using Composite Score ===
        if not results or not metrics_per_model:
            context["forecast"]["model_used"] = "none"
            context["forecast"]["predicted_movement"] = "error"
            context["forecast"]["confidence"] = 0.0
            context["explainability"]["full_trace"].append("[ForecastAgent] All models failed.")
            return context

        # Select best model using composite score
        model_scores = {}
        for model_name, metrics in metrics_per_model.items():
            score = self.composite_score(metrics, latest_price)
            model_scores[model_name] = score

        best_model = max(model_scores.items(), key=lambda x: x[1])[0]
        forecast_value = results[best_model]
        best_metrics = metrics_per_model[best_model]

        # Calculate movement and confidence
        movement = "up" if forecast_value > latest_price else "down" if forecast_value < latest_price else "stable"

        # Enhanced confidence calculation using directional accuracy
        directional_conf = best_metrics["Directional_Accuracy"]
        rmse_conf = max(0.0, min(1.0, 1 - (best_metrics["RMSE"] / max(latest_price, 1e-8))))
        confidence = round((0.6 * directional_conf + 0.4 * rmse_conf), 3)

        # === Update context ===
        context["forecast"]["model_used"] = best_model
        context["forecast"]["predicted_movement"] = movement
        context["forecast"]["confidence"] = confidence
        context["forecast"]["metrics_per_model"] = metrics_per_model

        context["explainability"]["full_trace"].append(
            f"[ForecastAgent] {best_model} selected (composite score: {model_scores[best_model]:.3f}) "
            f"predicts {movement.upper()} (conf={confidence:.3f}). "
            f"Metrics: RMSE=${best_metrics['RMSE']:.2f}, MAE=${best_metrics['MAE']:.2f}, "
            f"MAPE={best_metrics['MAPE']:.1f}%, Dir_Acc={best_metrics['Directional_Accuracy']:.1%}"
        )
        return context

# -----------------------------
# 🧠 Enhanced SentimentAgent: NewsAPI + TextBlob + Advanced Metrics
# -----------------------------
from textblob import TextBlob
from newsapi import NewsApiClient
from datetime import datetime as _dt, timedelta
import re

class SentimentAgent(AgentBase):
    def __init__(self):
        api_key = os.environ.get('NEWS_API_KEY')
        if not api_key:
            raise ValueError("🛑 NEWS_API_KEY not found in environment variables.")
        self.newsapi = NewsApiClient(api_key=api_key)

    def calculate_sentiment_directional_accuracy(self, context):
        """Calculate historical sentiment directional accuracy"""
        sentiment_history = context.get("sentiment_history", [])

        if len(sentiment_history) < 2:
            return 0.65  # Default reasonable accuracy for bootstrap

        # Count correct directional predictions
        correct_predictions = 0
        total_predictions = 0

        for entry in sentiment_history[-10:]:  # Last 10 predictions
            if all(k in entry for k in ["sentiment_direction", "actual_direction"]):
                total_predictions += 1
                if entry["sentiment_direction"] == entry["actual_direction"]:
                    correct_predictions += 1

        if total_predictions == 0:
            return 0.65  # Default

        return round(correct_predictions / total_predictions, 3)

    def categorize_sentiment_strength(self, polarity_score):
        """Categorize sentiment strength based on absolute polarity"""
        abs_score = abs(polarity_score)
        if abs_score >= 0.6:
            return "Strong"
        elif abs_score >= 0.2:
            return "Moderate"
        else:
            return "Weak"

    def calculate_recency_score(self, articles):
        """Calculate how fresh the news is (0-1 scale)"""
        if not articles:
            return 0.0

        now = _dt.now()
        recency_scores = []

        for article in articles:
            published_at = article.get('publishedAt')
            if not published_at:
                continue

            try:
                # Parse ISO format: 2024-01-15T10:30:00Z
                pub_time = _dt.fromisoformat(published_at.replace('Z', '+00:00'))
                pub_time = pub_time.replace(tzinfo=None)  # Remove timezone for comparison

                # Calculate hours ago
                hours_ago = (now - pub_time).total_seconds() / 3600

                # Score: 1.0 for very recent (0-6hrs), decaying to 0.1 for old (48+ hrs)
                if hours_ago <= 6:
                    score = 1.0
                elif hours_ago <= 24:
                    score = 0.8
                elif hours_ago <= 48:
                    score = 0.5
                else:
                    score = 0.1

                recency_scores.append(score)
            except:
                recency_scores.append(0.3)  # Default for parsing errors

        return round(sum(recency_scores) / len(recency_scores) if recency_scores else 0.3, 3)

    def run(self, context):
        if not context["sentiment"]["enabled"]:
            return context

        # ✅ Dynamic crypto symbol mapping for news search
        symbol = context["data"]["symbol"]
        crypto_search_terms = {
            "BTC": "bitcoin",
            "ETH": "ethereum",
            "SOL": "solana",
            "ADA": "cardano",
            "XRP": "ripple",
            "BNB": "binance coin",
            "MATIC": "polygon",
            "DOT": "polkadot",
            "LINK": "chainlink",
            "AVAX": "avalanche",
            "UNI": "uniswap",
            "LTC": "litecoin",
            "ATOM": "cosmos",
            "FTM": "fantom",
            "ALGO": "algorand"
        }
        search_term = crypto_search_terms.get(symbol, symbol.lower())

        try:
            articles = self.newsapi.get_everything(
                q=search_term,
                language='en',
                sort_by='publishedAt',
                page_size=10  # Increased for better volume analysis
            )
        except Exception as e:
            context["sentiment"].update({
                "source": "NewsAPI",
                "score": 0.0,
                "verdict": "neutral",
                "confidence": 0.0,
                "strength_category": "Weak",
                "news_volume": 0,
                "recency_score": 0.0,
                "directional_accuracy": 0.65,
                "headlines_sample": []
            })
            context["explainability"]["full_trace"].append(
                f"[SentimentAgent] Error fetching '{search_term}' headlines: {e}. Defaulting to NEUTRAL."
            )
            return context

        articles_list = articles.get('articles', [])
        headlines = [a['title'] for a in articles_list if a.get('title')]

        if not headlines:
            context["sentiment"].update({
                "source": "NewsAPI",
                "score": 0.0,
                "verdict": "neutral",
                "confidence": 0.0,
                "strength_category": "Weak",
                "news_volume": 0,
                "recency_score": 0.0,
                "directional_accuracy": 0.65,
                "headlines_sample": []
            })
            context["explainability"]["full_trace"].append(
                f"[SentimentAgent] No '{search_term}' headlines found. Defaulting to NEUTRAL."
            )
            return context

        # ✅ ENHANCED SENTIMENT ANALYSIS

        # Basic sentiment analysis
        polarities = [TextBlob(title).sentiment.polarity for title in headlines]
        avg_polarity = float(sum(polarities) / len(polarities))
        std_dev = float(np.std(polarities))

        # ✅ NEW METRICS CALCULATION

        # 1. Strength categorization
        strength_category = self.categorize_sentiment_strength(avg_polarity)

        # 2. News volume
        news_volume = len(headlines)

        # 3. Recency score
        recency_score = self.calculate_recency_score(articles_list)

        # 4. Historical directional accuracy
        directional_accuracy = self.calculate_sentiment_directional_accuracy(context)

        # 5. Enhanced confidence calculation
        base_confidence = 1.0 - std_dev  # Agreement between headlines
        volume_boost = min(0.2, news_volume / 50)  # More articles = higher confidence
        recency_boost = recency_score * 0.15  # Fresh news = higher confidence

        enhanced_confidence = round(
            0.6 * base_confidence + 0.3 * volume_boost + 0.1 * recency_boost, 3
        )
        enhanced_confidence = max(0.0, min(1.0, enhanced_confidence))

        # Verdict determination
        if avg_polarity > 0.2:
            verdict = "bullish"
        elif avg_polarity < -0.2:
            verdict = "bearish"
        else:
            verdict = "neutral"

        # Sample headlines for transparency
        headlines_sample = headlines[:3]  # Show first 3 headlines

        # ✅ UPDATE CONTEXT WITH ALL NEW METRICS
        context["sentiment"].update({
            "source": "NewsAPI",
            "score": round(avg_polarity, 3),
            "verdict": verdict,
            "confidence": enhanced_confidence,
            "strength_category": strength_category,
            "news_volume": news_volume,
            "recency_score": recency_score,
            "directional_accuracy": directional_accuracy,
            "used_in_decision": False,  # updated by DecisionAgent
            "alignment": None,
            "headlines_sample": headlines_sample
        })

        if context["explainability"]["enabled"]:
            context["explainability"]["full_trace"].append(
                f"[SentimentAgent] Analyzed {news_volume} '{search_term}' headlines: "
                f"Avg polarity = {avg_polarity:.3f} ({strength_category} strength), "
                f"Recency = {recency_score:.1%}, Dir_Acc = {directional_accuracy:.1%} "
                f"→ {verdict.upper()} (Confidence={enhanced_confidence:.1%})"
            )
        return context

# -----------------------------
# ✅ DecisionAgent: symmetric buy/sell + sentiment-adjusted thresholds
# -----------------------------
class DecisionAgent(AgentBase):
    def run(self, context):
        forecast = context["forecast"]
        sentiment = context["sentiment"]

        action = "hold"
        rationale = ""
        used_sentiment = bool(sentiment.get("enabled", False))
        aligned = None

        move = forecast.get("predicted_movement", "stable")   # "up" / "down" / "stable"
        conf = float(forecast.get("confidence", 0.0))
        sv = sentiment.get("verdict", "neutral") if used_sentiment else "neutral"  # "bullish"/"bearish"/"neutral"

        # Base threshold
        THRESH = 0.60
        # Sentiment-adjusted thresholds
        if move == "up":
            aligned = (sv == "bullish")
            thr = THRESH - 0.10 if aligned else (THRESH + 0.10 if sv == "bearish" else THRESH)
        elif move == "down":
            aligned = (sv == "bearish")
            thr = THRESH - 0.10 if aligned else (THRESH + 0.10 if sv == "bullish" else THRESH)
        else:  # "stable"
            aligned = None
            thr = 0.75  # be conservative when model says "stable"

        # Decision
        if move == "up" and conf >= thr:
            action = "buy"
            rationale = f"Forecast {move.upper()} with confidence {conf:.3f} (threshold {thr:.2f}); sentiment={sv}."
        elif move == "down" and conf >= thr:
            action = "sell"
            rationale = f"Forecast {move.upper()} with confidence {conf:.3f} (threshold {thr:.2f}); sentiment={sv}."
        else:
            reason = "confidence too low" if conf < thr else "no directional edge"
            action = "hold"
            rationale = f"Holding: {reason} (conf {conf:.3f} vs thresh {thr:.2f}); sentiment={sv}."

        # ✅ Save result
        context["decision"]["action"] = action
        context["decision"]["rationale"] = rationale

        # ✅ Mark sentiment usage + expose alignment for UI/prints
        context["sentiment"]["used_in_decision"] = used_sentiment
        context["sentiment"]["alignment"] = aligned

        # ✅ Store correlation for analysis
        context["history"].append({
            "timestamp": context["data"]["timestamp"],
            "forecast_movement": move,
            "forecast_confidence": conf,
            "sentiment_verdict": sv,
            "sentiment_score": sentiment.get("score"),
            "sentiment_confidence": sentiment.get("confidence"),
            "sentiment_strength": sentiment.get("strength_category"),
            "news_volume": sentiment.get("news_volume"),
            "recency_score": sentiment.get("recency_score"),
            "alignment": aligned,
            "used_sentiment": used_sentiment
        })

        # ✅ Explainability trace
        if context["explainability"]["enabled"]:
            context["explainability"]["full_trace"].append(
                f"[DecisionAgent] Action: {action} → {rationale}"
            )
        return context

# -----------------------------
# 🔧 Legacy helpers (kept for compatibility)
# -----------------------------
from datetime import datetime

def fetch_price_data(context, symbol="BTC"):
    df = yf.download(tickers=symbol + "-USD", period="120d", interval="1d", progress=False)
    context["data"]["symbol"] = symbol
    context["data"]["df"] = df
    context["data"]["timestamp"] = df.index[-1].strftime("%Y-%m-%d")
    context["explainability"]["full_trace"].append(
        f"[Data Fetch] Loaded price data for {symbol} at {context['data']['timestamp']}"
    )
    return context

def finalize_and_log(context):
    context["history"].append({
        "timestamp": context["data"].get("timestamp", str(datetime.today().date())),
        "decision": context.get("decision", {}),
        "forecast": context.get("forecast", {}),
        "sentiment": context["sentiment"] if context.get("sentiment", {}).get("enabled") else "N/A",
        "explanation_trace": context["explainability"].get("full_trace", [])
    })
    return context

In [ ]:

# 📦 MCP-Tradi-Win - Cell 4: Simplified Enhanced Pipeline Runner
# Fallback runner for direct agent execution without orchestrator

# -----------------------------
# 🎯 Enhanced Pipeline Runner (Simplified - No Complex Orchestrator)
# -----------------------------
def MCP_tradi_win_runner(context, use_sentiment=False, model_preference=None):
    """
    Simplified MCP-Tradi-Win pipeline runner for direct execution:

    Args:
        context: Shared context dictionary
        use_sentiment: Enable sentiment analysis (default: False)
        model_preference: Force specific model selection ('ARIMA', 'Prophet', 'LSTM', or None for auto)

    Returns:
        Updated context with all analysis results and full trace

    Note: For advanced orchestration, use OrchestratorChain in Cell 6
    """

    # Initialize configuration
    context["sentiment"]["enabled"] = use_sentiment
    start_time = pd.Timestamp.now()

    context["explainability"]["full_trace"].append(
        f"[SimplePipeline] Starting analysis at {start_time.strftime('%H:%M:%S')} "
        f"(Sentiment: {'Enabled' if use_sentiment else 'Disabled'})"
    )

    try:
        # Step 1: Data fetching with error handling
        try:
            context = fetch_price_data(context)
            context["explainability"]["full_trace"].append(
                "[SimplePipeline] ✅ Data fetch completed successfully"
            )
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[SimplePipeline] ❌ Data fetch failed: {e}"
            )
            return context

        # Step 2: Forecasting with error handling
        try:
            forecast_agent = ForecastAgent()
            context = forecast_agent.run(context)

            # Apply model preference if specified
            if model_preference and model_preference in context.get("forecast", {}).get("metrics_per_model", {}):
                original_model = context["forecast"]["model_used"]

                # Update forecast results to use preferred model
                metrics = context["forecast"]["metrics_per_model"][model_preference]
                context["forecast"]["model_used"] = model_preference

                # Recalculate confidence for preferred model
                latest_price = 50000  # Approximate for confidence calculation
                directional_conf = metrics["Directional_Accuracy"]
                rmse_conf = max(0.0, min(1.0, 1 - (metrics["RMSE"] / latest_price)))
                new_confidence = round((0.6 * directional_conf + 0.4 * rmse_conf), 3)
                context["forecast"]["confidence"] = new_confidence

                context["explainability"]["full_trace"].append(
                    f"[SimplePipeline] Model preference override: {original_model} → {model_preference} "
                    f"(Dir_Acc: {metrics['Directional_Accuracy']:.1%}, New confidence: {new_confidence:.1%})"
                )

            context["explainability"]["full_trace"].append(
                "[SimplePipeline] ✅ Forecasting completed successfully"
            )
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[SimplePipeline] ❌ Forecasting failed: {e}"
            )
            # Set error state but continue
            context["forecast"] = {"model_used": "error", "predicted_movement": "error", "confidence": 0.0}

        # Step 3: Sentiment analysis with error handling
        try:
            if use_sentiment:
                sentiment_agent = SentimentAgent()
                context = sentiment_agent.run(context)
                context["explainability"]["full_trace"].append(
                    "[SimplePipeline] ✅ Sentiment analysis completed successfully"
                )
            else:
                context["explainability"]["full_trace"].append(
                    "[SimplePipeline] ⏭️ Sentiment analysis skipped (disabled)"
                )
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[SimplePipeline] ❌ Sentiment analysis failed: {e} - continuing with neutral sentiment"
            )
            # Ensure neutral sentiment for decision making
            context["sentiment"].update({
                "source": "Error_Fallback",
                "score": 0.0,
                "verdict": "neutral",
                "confidence": 0.0,
                "strength_category": "Weak",
                "news_volume": 0,
                "recency_score": 0.0,
                "directional_accuracy": 0.65
            })

        # Step 4: Decision making with error handling
        try:
            decision_agent = DecisionAgent()
            context = decision_agent.run(context)
            context["explainability"]["full_trace"].append(
                "[SimplePipeline] ✅ Decision making completed successfully"
            )
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[SimplePipeline] ❌ Decision making failed: {e}"
            )
            # Set safe default decision
            context["decision"] = {"action": "hold", "rationale": f"Error in decision process: {e}"}

    except Exception as e:
        # Global error handler
        context["explainability"]["full_trace"].append(
            f"[SimplePipeline] ❌ Critical pipeline failure: {e}"
        )
        # Ensure minimal viable context
        if "decision" not in context:
            context["decision"] = {"action": "hold", "rationale": "Pipeline error - defaulting to hold"}

    # Final logging and timing
    try:
        context = finalize_and_log(context)
        end_time = pd.Timestamp.now()
        execution_time = (end_time - start_time).total_seconds()

        context["explainability"]["full_trace"].append(
            f"[SimplePipeline] ✅ Analysis completed in {execution_time:.2f}s at {end_time.strftime('%H:%M:%S')} "
            f"→ Action: {context.get('decision', {}).get('action', 'unknown').upper()}"
        )
    except Exception as e:
        context["explainability"]["full_trace"].append(
            f"[SimplePipeline] ⚠️ Logging failed: {e} - but analysis completed"
        )

    return context

# -----------------------------
# 🎲 Quick Test Runner for Development (Updated)
# -----------------------------
def quick_test_runner(symbol="BTC", use_sentiment=False, model_preference=None):
    """
    Quick test function for development and debugging

    Args:
        symbol: Crypto symbol to analyze
        use_sentiment: Enable sentiment analysis
        model_preference: Force specific model ('ARIMA', 'Prophet', 'LSTM')
    """
    test_context = context.copy()
    test_context["data"]["symbol"] = symbol

    print(f"🧪 Quick Test: {symbol} | Sentiment: {use_sentiment} | Model: {model_preference or 'Auto'}")
    print("-" * 70)

    result = MCP_tradi_win_runner(
        test_context,
        use_sentiment=use_sentiment,
        model_preference=model_preference
    )

    # Print key results
    decision = result.get("decision", {})
    forecast = result.get("forecast", {})
    sentiment = result.get("sentiment", {})

    print(f"📊 Model: {forecast.get('model_used', 'N/A')}")
    print(f"📈 Prediction: {forecast.get('predicted_movement', 'N/A')} (confidence: {forecast.get('confidence', 0):.1%})")

    if sentiment.get("enabled"):
        print(f"🧠 Sentiment: {sentiment.get('verdict', 'N/A')} ({sentiment.get('strength_category', 'N/A')})")
        print(f"📰 News: {sentiment.get('news_volume', 0)} articles, {sentiment.get('recency_score', 0):.1%} fresh")

    print(f"💡 Decision: {decision.get('action', 'N/A')}")
    print(f"🧠 Rationale: {decision.get('rationale', 'N/A')}")

    return result

In [ ]:

# Cell 5 - Enhanced Demo with Advanced Sentiment Metrics Display

# Install required packages
!pip install tabulate --quiet

from tabulate import tabulate
from datetime import datetime

# ✅ Updated context with today's date
context = {
    "data": {
        "symbol": "BTC",
        "timestamp": datetime.today().strftime("%Y-%m-%d"),  # ✅ today's date
    },
    "forecast": {},
    "sentiment": {
        "enabled": True
    },
    "decision": {},
    "explainability": {
        "enabled": True,              # ✅ ADD THIS LINE
        "full_trace": []
    },
    "history": [],
    "sentiment_history": []  # ✅ NEW: For sentiment tracking
}

# 1. Run the full pipeline
final_context = MCP_tradi_win_runner(context, use_sentiment=True)

# 2. Display final decision
print("══════════════════════════════════════════════════")
print("📈 FINAL TRADING DECISION")
print("══════════════════════════════════════════════════")
print(f"🔹 Action: {final_context['decision']['action'].upper()}")
print(f"🔹 Rationale: {final_context['decision']['rationale']}\n")

# 3. Enhanced forecast info with focus on our key metrics
print("🔍 FORECAST DETAILS")
print(f"🔹 Model Used: {final_context['forecast']['model_used']} (selected by composite score)")
print(f"🔹 Predicted Movement: {final_context['forecast']['predicted_movement'].upper()}")
print(f"🔹 Confidence Score: {final_context['forecast']['confidence']:.1%}")

if "metrics_per_model" in final_context["forecast"]:
    best_model = final_context["forecast"]["model_used"]
    best_metrics = final_context["forecast"]["metrics_per_model"].get(best_model, {})

    print(f"🎯 Key Performance Metrics:")
    print(f"   • Directional Accuracy: {best_metrics.get('Directional_Accuracy', 0):.1%} (most important for trading)")
    print(f"   • MAPE: {best_metrics.get('MAPE', 0):.1f}% (scale-independent error)")
    print(f"   • MAE: ${best_metrics.get('MAE', 0):.2f} (absolute error magnitude)")
    print(f"   • RMSE: ${best_metrics.get('RMSE', 0):.2f} (reference metric)")
    print()

# 4. ✅ ENHANCED SENTIMENT ANALYSIS DISPLAY
if final_context["sentiment"]["enabled"]:
    print("🧠 ADVANCED SENTIMENT ANALYSIS")
    print(f"🔹 Verdict: {final_context['sentiment']['verdict'].upper()} ({final_context['sentiment']['strength_category']} Strength)")
    print(f"🔹 Score: {final_context['sentiment']['score']:.3f} (range: -1 to +1)")

    # ✅ NEW METRICS DISPLAY
    print(f"📊 Sentiment Performance Metrics:")
    print(f"   • News Volume: {final_context['sentiment']['news_volume']} articles analyzed")
    print(f"   • News Freshness: {final_context['sentiment']['recency_score']:.1%} (recent news weight)")
    print(f"   • Historical Accuracy: {final_context['sentiment']['directional_accuracy']:.1%} (directional hit rate)")
    print(f"   • Enhanced Confidence: {final_context['sentiment'].get('confidence', 0):.1%} (volume + agreement + recency)")

    # Alignment display
    alignment_status = final_context['sentiment'].get('alignment')
    if alignment_status is True:
        print(f"🔹 Forecast-Sentiment Alignment: ✅ ALIGNED (threshold boost)")
    elif alignment_status is False:
        print(f"🔹 Forecast-Sentiment Alignment: ❌ CONFLICTING (threshold penalty)")
    else:
        print(f"🔹 Forecast-Sentiment Alignment: ➖ NEUTRAL (no adjustment)")

    print(f"🔹 Used in Decision: {'✅ YES' if final_context['sentiment'].get('used_in_decision') else '❌ NO'}")

    # ✅ SAMPLE HEADLINES FOR TRANSPARENCY
    headlines = final_context['sentiment'].get('headlines_sample', [])
    if headlines:
        print(f"📰 Sample Headlines Analyzed:")
        for i, headline in enumerate(headlines, 1):
            print(f"   {i}. {headline[:80]}{'...' if len(headline) > 80 else ''}")
    print()

# 5. Explainability trace
print("🛡️ EXPLAINABILITY TRACE")
for i, step in enumerate(final_context["explainability"]["full_trace"], 1):
    print(f"{i:02d}. {step}")
print()

# 6. 📊 Enhanced Forecast Metrics Table with Directional Accuracy Focus
if "metrics_per_model" in final_context["forecast"]:
    print("📊 MODEL PERFORMANCE COMPARISON")
    print("🎯 Ranked by Composite Score: 40% Directional Accuracy + 30% MAPE + 30% MAE")

    model_table = []
    selected_model = final_context["forecast"]["model_used"]

    # Calculate composite scores for ranking
    model_scores = []
    for model_name, metrics in final_context["forecast"]["metrics_per_model"].items():
        # Recreate composite score calculation (should match ForecastAgent logic)
        latest_price = 50000  # Approximate, for display purposes
        dir_acc_score = metrics.get("Directional_Accuracy", 0)
        mape_score = max(0, 1 - metrics.get("MAPE", 0) / 100)
        normalized_mae = metrics.get("MAE", 0) / latest_price
        mae_score = max(0, 1 - normalized_mae)
        composite = 0.4 * dir_acc_score + 0.3 * mape_score + 0.3 * mae_score

        model_scores.append((model_name, composite, metrics))

    # Sort by composite score (descending)
    model_scores.sort(key=lambda x: x[1], reverse=True)

    for rank, (model_name, composite_score, metrics) in enumerate(model_scores, 1):
        selected_indicator = "👑" if model_name == selected_model else f"{rank}."
        model_table.append([
            f"{selected_indicator} {model_name}",
            f"{metrics.get('Directional_Accuracy', 0):.1%}",
            f"{metrics.get('MAPE', 0):.1f}%",
            f"${metrics.get('MAE', 0):.2f}",
            f"${metrics.get('RMSE', 0):.2f}",
            f"{composite_score:.3f}"
        ])

    print(tabulate(
        model_table,
        headers=["Model", "Dir Acc ↑", "MAPE ↓", "MAE ↓", "RMSE ↓", "Score ↑"],
        tablefmt="fancy_grid"
    ))
    print("Legend: ↑ = Higher is better, ↓ = Lower is better, 👑 = Selected model")
    print()

# ✅ 7. NEW SENTIMENT PERFORMANCE TABLE
if final_context["sentiment"]["enabled"]:
    print("🧠 SENTIMENT ANALYSIS BREAKDOWN")

    sentiment_performance = [
        ["Verdict", final_context['sentiment']['verdict'].upper()],
        ["Strength Category", final_context['sentiment']['strength_category']],
        ["Polarity Score", f"{final_context['sentiment']['score']:.3f}"],
        ["News Volume", f"{final_context['sentiment']['news_volume']} articles"],
        ["News Freshness", f"{final_context['sentiment']['recency_score']:.1%}"],
        ["Historical Accuracy", f"{final_context['sentiment']['directional_accuracy']:.1%}"],
        ["Enhanced Confidence", f"{final_context['sentiment']['confidence']:.1%}"]
    ]

    print(tabulate(
        sentiment_performance,
        headers=["Metric", "Value"],
        tablefmt="fancy_grid"
    ))
    print()

# 8. 📈 Enhanced Correlation History with New Metrics
if "history" in final_context and final_context["history"]:
    print("📈 COMPREHENSIVE DECISION ANALYSIS HISTORY")
    corr_table = []

    for h in final_context["history"]:
        # Skip any entries missing required fields
        required_keys = ["timestamp", "forecast_movement", "forecast_confidence", "sentiment_verdict"]
        if not all(k in h for k in required_keys):
            continue

        # Format alignment with symbols
        alignment_symbol = "✅" if h.get("alignment") is True else "❌" if h.get("alignment") is False else "➖"

        corr_table.append([
            h["timestamp"],
            h["forecast_movement"].upper(),
            f"{h['forecast_confidence']:.1%}",
            h["sentiment_verdict"].upper(),
            h.get("sentiment_strength", "N/A"),
            h.get("news_volume", "N/A"),
            f"{h.get('recency_score', 0):.1%}",
            alignment_symbol,
            "✅" if h.get("used_sentiment") else "❌"
        ])

    if corr_table:
        print(tabulate(
            corr_table,
            headers=["Date", "Forecast", "F_Conf", "Sentiment", "Strength", "Volume", "Fresh", "Align", "Used"],
            tablefmt="fancy_grid"
        ))
        print("Legend: Align: ✅=Aligned, ❌=Conflicting, ➖=Neutral | Fresh=News Freshness")
    else:
        print("No valid correlation entries available.")

print("\n" + "="*80)
print("🎓 MCP-TRADI-WIN: Explainable Multi-Agent Crypto Trading Assistant")
print("   Enhanced Focus: Directional Accuracy + MAPE + MAE + Advanced Sentiment Metrics")
print("   📊 Forecast: Dir_Acc (40%) + MAPE (30%) + MAE (30%)")
print("   🧠 Sentiment: Volume + Freshness + Historical Accuracy + Agreement")
print("="*80)

✅ Latest Fetched Date: 2025-10-29
══════════════════════════════════════════════════
📈 FINAL TRADING DECISION
══════════════════════════════════════════════════
🔹 Action: HOLD
🔹 Rationale: Holding: confidence too low (conf 0.000 vs thresh 0.75); sentiment=neutral.

🔍 FORECAST DETAILS
🔹 Model Used: none (selected by composite score)
🔹 Predicted Movement: ERROR
🔹 Confidence Score: 0.0%
🧠 ADVANCED SENTIMENT ANALYSIS
🔹 Verdict: NEUTRAL (Weak Strength)
🔹 Score: 0.000 (range: -1 to +1)
📊 Sentiment Performance Metrics:
   • News Volume: 0 articles analyzed
   • News Freshness: 0.0% (recent news weight)
   • Historical Accuracy: 65.0% (directional hit rate)
   • Enhanced Confidence: 0.0% (volume + agreement + recency)
🔹 Forecast-Sentiment Alignment: ➖ NEUTRAL (no adjustment)
🔹 Used in Decision: ✅ YES

🛡️ EXPLAINABILITY TRACE
01. [SimplePipeline] Starting analysis at 08:03:54 (Sentiment: Enabled)
02. [Data Fetch] Loaded price data for BTC at 2025-10-29
03. [SimplePipeline] ✅ Data fetch complete

In [ ]:

# Cell 6 – Enhanced OrchestratorChain + Advanced Sentiment Persistence
# This cell introduces the practical Agentic AI orchestration layer, replacing complex orchestrators
# with a modular OrchestratorChain that runs each agent (DataFetcher → Forecast → Sentiment → Decision) in sequence.
# Enhanced with advanced sentiment metrics persistence and comprehensive artifact storage.

# === Enhanced OrchestratorChain + DataFetcherAgent + Advanced Persistence ===
import json, os, csv
from datetime import datetime

ART_DIR = "artifacts"
os.makedirs(ART_DIR, exist_ok=True)

# Wrap your existing fetch function as an agent
class DataFetcherAgent(AgentBase):
    def run(self, context):
        return fetch_price_data(context, symbol=context["data"].get("symbol","BTC"))

def persist_artifacts(context):
    """
    Enhanced persistence function with advanced sentiment metrics support
    Saves JSON artifacts and CSV correlation log with all new metrics
    """

    # ✅ Enhanced JSON artifacts with all sentiment metrics
    if context.get("forecast"):
        forecast_data = context["forecast"].copy()
        # Add timestamp for tracking
        forecast_data["timestamp"] = context["data"].get("timestamp")
        forecast_data["symbol"] = context["data"].get("symbol")

        with open(f"{ART_DIR}/forecast_results.json", "w") as f:
            json.dump(forecast_data, f, indent=2)

    if context.get("sentiment"):
        sentiment_data = context["sentiment"].copy()
        # Add timestamp for tracking
        sentiment_data["timestamp"] = context["data"].get("timestamp")
        sentiment_data["symbol"] = context["data"].get("symbol")

        with open(f"{ART_DIR}/sentiment_results.json", "w") as f:
            json.dump(sentiment_data, f, indent=2, default=str)  # default=str for any complex objects

    if context.get("decision"):
        decision_data = context["decision"].copy()
        # Add context for decision tracking
        decision_data["timestamp"] = context["data"].get("timestamp")
        decision_data["symbol"] = context["data"].get("symbol")
        decision_data["forecast_confidence"] = context.get("forecast", {}).get("confidence")
        decision_data["sentiment_enabled"] = context.get("sentiment", {}).get("enabled", False)

        with open(f"{ART_DIR}/decision.json", "w") as f:
            json.dump(decision_data, f, indent=2)

    # ✅ Enhanced trace append with more context
    if context.get("explainability", {}).get("full_trace"):
        with open(f"{ART_DIR}/trace.jsonl", "a") as f:
            trace_entry = {
                "timestamp": datetime.utcnow().isoformat() + "Z",
                "symbol": context["data"].get("symbol", "BTC"),
                "actor": "Pipeline",
                "trace_step": context["explainability"]["full_trace"][-1],
                "forecast_model": context.get("forecast", {}).get("model_used"),
                "sentiment_enabled": context.get("sentiment", {}).get("enabled", False)
            }
            f.write(json.dumps(trace_entry) + "\n")

    # ✅ Enhanced correlation log CSV with ALL advanced sentiment metrics
    try:
        hist = context.get("history", [])
        if hist:
            path = f"{ART_DIR}/correlation_log.csv"
            file_exists = os.path.exists(path)

            # ✅ ENHANCED FIELDNAMES with all new sentiment metrics
            fieldnames = [
                # Basic info
                "timestamp", "symbol",
                # Forecast metrics
                "forecast_movement", "forecast_confidence", "forecast_model",
                # Enhanced sentiment metrics
                "sentiment_verdict", "sentiment_score", "sentiment_confidence",
                "sentiment_strength_category", "news_volume", "recency_score",
                "sentiment_directional_accuracy",
                # Decision metrics
                "alignment", "used_sentiment", "decision_action"
            ]

            with open(path, "a", newline="") as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()

                # Write only the last row from history with all enhanced metrics
                last = hist[-1]
                sentiment = context.get("sentiment", {})
                forecast = context.get("forecast", {})
                decision = context.get("decision", {})

                writer.writerow({
                    # Basic info
                    "timestamp": context["data"].get("timestamp"),
                    "symbol": context["data"].get("symbol", "BTC"),

                    # Forecast metrics
                    "forecast_movement": last.get("forecast_movement"),
                    "forecast_confidence": last.get("forecast_confidence"),
                    "forecast_model": forecast.get("model_used"),

                    # Enhanced sentiment metrics
                    "sentiment_verdict": last.get("sentiment_verdict"),
                    "sentiment_score": last.get("sentiment_score"),
                    "sentiment_confidence": last.get("sentiment_confidence"),
                    "sentiment_strength_category": sentiment.get("strength_category"),
                    "news_volume": sentiment.get("news_volume"),
                    "recency_score": sentiment.get("recency_score"),
                    "sentiment_directional_accuracy": sentiment.get("directional_accuracy"),

                    # Decision metrics
                    "alignment": last.get("alignment"),
                    "used_sentiment": last.get("used_sentiment"),
                    "decision_action": decision.get("action")
                })

    except Exception as e:
        # Non-fatal; continue pipeline
        print(f"Warning: CSV persistence failed: {e}")
        pass

    # ✅ NEW: Save model performance metrics for analysis
    try:
        if context.get("forecast", {}).get("metrics_per_model"):
            metrics_data = {
                "timestamp": context["data"].get("timestamp"),
                "symbol": context["data"].get("symbol"),
                "selected_model": context["forecast"].get("model_used"),
                "metrics": context["forecast"]["metrics_per_model"]
            }

            with open(f"{ART_DIR}/model_performance.json", "w") as f:
                json.dump(metrics_data, f, indent=2)

    except Exception as e:
        # Non-fatal
        pass

class OrchestratorChain:
    """
    Enhanced practical MCP orchestrator for crypto trading pipeline
    Executes agents sequentially with comprehensive error handling and persistence
    """

    def __init__(self, agents, persist=None):
        self.agents = agents  # [DataFetcherAgent(), ForecastAgent(), SentimentAgent(), DecisionAgent()]
        self.persist = persist

    def run(self, context):
        """
        Execute the complete agent pipeline with enhanced error handling
        """
        successful_agents = 0
        total_agents = len(self.agents)

        context["explainability"]["full_trace"].append(
            f"[OrchestratorChain] Starting pipeline with {total_agents} agents"
        )

        for i, agent in enumerate(self.agents, 1):
            agent_name = agent.__class__.__name__

            try:
                # Execute agent
                context = agent.run(context)
                successful_agents += 1

                # Log success
                if context.get("explainability", {}).get("enabled"):
                    context["explainability"]["full_trace"].append(
                        f"[OrchestratorChain] ✅ {agent_name} completed ({i}/{total_agents})"
                    )

                # Persist after each successful agent (for incremental backup)
                if self.persist:
                    self.persist(context)

            except Exception as e:
                # Log failure but continue pipeline where possible
                error_msg = f"[OrchestratorChain] ❌ {agent_name} failed: {e}"
                context["explainability"]["full_trace"].append(error_msg)

                # For critical agents, might want to stop pipeline
                if agent_name in ["DataFetcherAgent", "ForecastAgent"]:
                    context["explainability"]["full_trace"].append(
                        f"[OrchestratorChain] 🛑 Critical agent {agent_name} failed - stopping pipeline"
                    )
                    break
                else:
                    # Non-critical agents can fail gracefully
                    context["explainability"]["full_trace"].append(
                        f"[OrchestratorChain] ⚠️ Non-critical agent {agent_name} failed - continuing pipeline"
                    )

        # Final pipeline summary
        context["explainability"]["full_trace"].append(
            f"[OrchestratorChain] Pipeline completed: {successful_agents}/{total_agents} agents successful"
        )

        # Final log and persistence
        try:
            context = finalize_and_log(context)
            if self.persist:
                self.persist(context)  # Final save
        except Exception as e:
            context["explainability"]["full_trace"].append(
                f"[OrchestratorChain] ⚠️ Final logging failed: {e}"
            )

        return context

# ✅ Enhanced convenience function for quick testing
def run_enhanced_pipeline(symbol="BTC", use_sentiment=True, save_artifacts=True):
    """
    Convenience function to run the complete enhanced pipeline
    Perfect for testing and development
    """

    # Fresh context
    test_context = context.copy()
    test_context["data"]["symbol"] = symbol
    test_context["sentiment"]["enabled"] = use_sentiment

    # Initialize agents
    agents = [
        DataFetcherAgent(),
        ForecastAgent(),
        SentimentAgent(),
        DecisionAgent()
    ]

    # Initialize orchestrator with optional persistence
    orchestrator = OrchestratorChain(
        agents=agents,
        persist=persist_artifacts if save_artifacts else None
    )

    # Execute pipeline
    print(f"🚀 Running enhanced pipeline for {symbol} (Sentiment: {use_sentiment})")
    print("-" * 60)

    result = orchestrator.run(test_context)

    # Quick summary
    decision = result.get("decision", {})
    forecast = result.get("forecast", {})
    sentiment = result.get("sentiment", {})

    print(f"✅ Pipeline completed!")
    print(f"📊 Model: {forecast.get('model_used', 'N/A')}")
    print(f"📈 Prediction: {forecast.get('predicted_movement', 'N/A')} ({forecast.get('confidence', 0):.1%})")

    if sentiment.get("enabled"):
        print(f"🧠 Sentiment: {sentiment.get('verdict', 'N/A')} ({sentiment.get('strength_category', 'N/A')} strength)")
        print(f"📰 News: {sentiment.get('news_volume', 0)} articles, {sentiment.get('recency_score', 0):.1%} fresh")

    print(f"💡 Decision: {decision.get('action', 'N/A')}")

    if save_artifacts:
        print(f"💾 Artifacts saved to {ART_DIR}/ folder")

    return result

In [ ]:

# Cell 7 – Enhanced Run the Agentic Chain with Advanced Sentiment Metrics
# This cell creates a fresh context and uses the enhanced OrchestratorChain from Cell 6 to execute
# the full pipeline with modular agents, displaying all advanced sentiment and forecast metrics.
# Perfect for testing the complete enhanced system end-to-end.

# === Enhanced Agentic Chain Execution with Advanced Metrics Display ===
from datetime import datetime
from tabulate import tabulate

print("🚀 TESTING ENHANCED AGENTIC ORCHESTRATOR CHAIN")
print("=" * 70)

# Fresh context (enhanced with all new fields)
final_chain_ctx = None
chain_context = {
    "data": {
        "symbol": "BTC",
        "timestamp": datetime.today().strftime("%Y-%m-%d"),
    },
    "forecast": {},
    "sentiment": {"enabled": True},
    "decision": {},
    "explainability": {"enabled": True, "full_trace": []},
    "history": [],
    "sentiment_history": []  # ✅ Enhanced with sentiment tracking
}

# Build and run enhanced chain
chain = OrchestratorChain(
    agents=[DataFetcherAgent(), ForecastAgent(), SentimentAgent(), DecisionAgent()],
    persist=persist_artifacts
)

print("⏳ Executing enhanced pipeline...")
final_chain_ctx = chain.run(chain_context)
print("✅ Pipeline completed!\n")

# === ENHANCED SUMMARY DISPLAY ===

# 1. Final Decision
print("══════════════════════════════════════════════════")
print("📈 FINAL TRADING DECISION (Enhanced Agentic Chain)")
print("══════════════════════════════════════════════════")
print(f"🔹 Action: {final_chain_ctx['decision'].get('action', 'N/A').upper()}")
print(f"🔹 Rationale: {final_chain_ctx['decision'].get('rationale', 'N/A')}\n")

# 2. Enhanced Forecast Details
print("🔍 ENHANCED FORECAST DETAILS")
print(f"🔹 Model Used: {final_chain_ctx['forecast'].get('model_used', 'N/A')} (selected by composite score)")
print(f"🔹 Predicted Movement: {final_chain_ctx['forecast'].get('predicted_movement', 'N/A').upper()}")
print(f"🔹 Confidence Score: {final_chain_ctx['forecast'].get('confidence', 0):.1%}")

mpm = final_chain_ctx["forecast"].get("metrics_per_model", {})
bm = final_chain_ctx['forecast'].get('model_used')
if bm and bm in mpm:
    best_metrics = mpm[bm]
    print(f"🎯 Key Performance Metrics:")
    print(f"   • Directional Accuracy: {best_metrics.get('Directional_Accuracy', 0):.1%} (most important for trading)")
    print(f"   • MAPE: {best_metrics.get('MAPE', 0):.1f}% (scale-independent error)")
    print(f"   • MAE: ${best_metrics.get('MAE', 0):.2f} (absolute error magnitude)")
    print(f"   • RMSE: ${best_metrics.get('RMSE', 0):.2f} (reference metric)")
print()

# 3. ✅ ENHANCED SENTIMENT ANALYSIS DISPLAY
if final_chain_ctx["sentiment"].get("enabled"):
    print("🧠 ENHANCED SENTIMENT ANALYSIS")
    print(f"🔹 Verdict: {final_chain_ctx['sentiment'].get('verdict', 'N/A').upper()} ({final_chain_ctx['sentiment'].get('strength_category', 'N/A')} Strength)")
    print(f"🔹 Score: {final_chain_ctx['sentiment'].get('score', 0):.3f} (range: -1 to +1)")

    # ✅ NEW ADVANCED METRICS DISPLAY
    print(f"📊 Sentiment Performance Metrics:")
    print(f"   • News Volume: {final_chain_ctx['sentiment'].get('news_volume', 0)} articles analyzed")
    print(f"   • News Freshness: {final_chain_ctx['sentiment'].get('recency_score', 0):.1%} (recent news weight)")
    print(f"   • Historical Accuracy: {final_chain_ctx['sentiment'].get('directional_accuracy', 0):.1%} (directional hit rate)")
    print(f"   • Enhanced Confidence: {final_chain_ctx['sentiment'].get('confidence', 0):.1%} (volume + agreement + recency)")

    # Alignment display
    alignment_status = final_chain_ctx['sentiment'].get('alignment')
    if alignment_status is True:
        print(f"🔹 Forecast-Sentiment Alignment: ✅ ALIGNED (threshold boost)")
    elif alignment_status is False:
        print(f"🔹 Forecast-Sentiment Alignment: ❌ CONFLICTING (threshold penalty)")
    else:
        print(f"🔹 Forecast-Sentiment Alignment: ➖ NEUTRAL (no adjustment)")

    print(f"🔹 Used in Decision: {'✅ YES' if final_chain_ctx['sentiment'].get('used_in_decision') else '❌ NO'}")

    # ✅ SAMPLE HEADLINES FOR TRANSPARENCY
    headlines = final_chain_ctx['sentiment'].get('headlines_sample', [])
    if headlines:
        print(f"📰 Sample Headlines Analyzed:")
        for i, headline in enumerate(headlines, 1):
            print(f"   {i}. {headline[:80]}{'...' if len(headline) > 80 else ''}")
    print()

# 4. Explainability Trace
print("🛡️ EXPLAINABILITY TRACE")
for i, step in enumerate(final_chain_ctx["explainability"]["full_trace"], 1):
    print(f"{i:02d}. {step}")
print()

# 5. ✅ ENHANCED FORECAST METRICS TABLE with Directional Accuracy
if mpm:
    print("📊 ENHANCED MODEL PERFORMANCE COMPARISON")
    print("🎯 Ranked by Composite Score: 40% Directional Accuracy + 30% MAPE + 30% MAE")

    model_table = []
    selected_model = final_chain_ctx["forecast"].get("model_used")

    # Calculate composite scores for ranking
    model_scores = []
    for model_name, metrics in mpm.items():
        # Recreate composite score calculation
        latest_price = 50000  # Approximate, for display purposes
        dir_acc_score = metrics.get("Directional_Accuracy", 0)
        mape_score = max(0, 1 - metrics.get("MAPE", 0) / 100)
        normalized_mae = metrics.get("MAE", 0) / latest_price
        mae_score = max(0, 1 - normalized_mae)
        composite = 0.4 * dir_acc_score + 0.3 * mape_score + 0.3 * mae_score

        model_scores.append((model_name, composite, metrics))

    # Sort by composite score (descending)
    model_scores.sort(key=lambda x: x[1], reverse=True)

    for rank, (model_name, composite_score, metrics) in enumerate(model_scores, 1):
        selected_indicator = "👑" if model_name == selected_model else f"{rank}."
        model_table.append([
            f"{selected_indicator} {model_name}",
            f"{metrics.get('Directional_Accuracy', 0):.1%}",
            f"{metrics.get('MAPE', 0):.1f}%",
            f"${metrics.get('MAE', 0):.2f}",
            f"${metrics.get('RMSE', 0):.2f}",
            f"{composite_score:.3f}"
        ])

    print(tabulate(
        model_table,
        headers=["Model", "Dir Acc ↑", "MAPE ↓", "MAE ↓", "RMSE ↓", "Score ↑"],
        tablefmt="fancy_grid"
    ))
    print("Legend: ↑ = Higher is better, ↓ = Lower is better, 👑 = Selected model")
    print()

# ✅ 6. NEW SENTIMENT PERFORMANCE BREAKDOWN TABLE
if final_chain_ctx["sentiment"].get("enabled"):
    print("🧠 SENTIMENT ANALYSIS BREAKDOWN")

    sentiment_performance = [
        ["Verdict", final_chain_ctx['sentiment'].get('verdict', 'N/A').upper()],
        ["Strength Category", final_chain_ctx['sentiment'].get('strength_category', 'N/A')],
        ["Polarity Score", f"{final_chain_ctx['sentiment'].get('score', 0):.3f}"],
        ["News Volume", f"{final_chain_ctx['sentiment'].get('news_volume', 0)} articles"],
        ["News Freshness", f"{final_chain_ctx['sentiment'].get('recency_score', 0):.1%}"],
        ["Historical Accuracy", f"{final_chain_ctx['sentiment'].get('directional_accuracy', 0):.1%}"],
        ["Enhanced Confidence", f"{final_chain_ctx['sentiment'].get('confidence', 0):.1%}"]
    ]

    print(tabulate(
        sentiment_performance,
        headers=["Metric", "Value"],
        tablefmt="fancy_grid"
    ))
    print()

# ✅ 7. ENHANCED CORRELATION HISTORY with All New Metrics
hist = final_chain_ctx.get("history", [])
if hist:
    print("📈 COMPREHENSIVE DECISION ANALYSIS HISTORY")
    corr_table = []

    for h in hist:
        # Skip any entries missing required fields
        required_keys = ["timestamp", "forecast_movement", "forecast_confidence", "sentiment_verdict"]
        if not all(k in h for k in required_keys):
            continue

        # Format alignment with symbols
        alignment_symbol = "✅" if h.get("alignment") is True else "❌" if h.get("alignment") is False else "➖"

        corr_table.append([
            h["timestamp"],
            h["forecast_movement"].upper(),
            f"{h['forecast_confidence']:.1%}",
            h["sentiment_verdict"].upper(),
            h.get("sentiment_strength", "N/A"),
            h.get("news_volume", "N/A"),
            f"{h.get('recency_score', 0):.1%}",
            alignment_symbol,
            "✅" if h.get("used_sentiment") else "❌"
        ])

    if corr_table:
        print(tabulate(
            corr_table,
            headers=["Date", "Forecast", "F_Conf", "Sentiment", "Strength", "Volume", "Fresh", "Align", "Used"],
            tablefmt="fancy_grid"
        ))
        print("Legend: Align: ✅=Aligned, ❌=Conflicting, ➖=Neutral | Fresh=News Freshness")
    else:
        print("No valid correlation entries available.")
else:
    print("📈 No correlation history available yet.")

# ✅ 8. ARTIFACTS SUMMARY
print(f"\n💾 ARTIFACTS PERSISTENCE")
print(f"📁 Location: {ART_DIR}/ folder")
print(f"📄 Files created:")
print(f"   • forecast_results.json (forecast metrics & model selection)")
print(f"   • sentiment_results.json (all advanced sentiment metrics)")
print(f"   • decision.json (trading decision & rationale)")
print(f"   • correlation_log.csv (historical analysis data)")
print(f"   • trace.jsonl (explainability trace log)")
print(f"   • model_performance.json (model comparison data)")

print("\n" + "="*80)
print("🎓 ENHANCED MCP-TRADI-WIN: Complete Agentic Pipeline Test")
print("   ✅ Advanced Sentiment Metrics: Volume + Freshness + Historical Accuracy")
print("   ✅ Enhanced Forecast Metrics: Directional Accuracy + MAPE + MAE")
print("   ✅ Comprehensive Persistence: JSON + CSV + JSONL artifacts")
print("   ✅ Full Explainability: Step-by-step trace with agent coordination")
print("="*80)

# ✅ 9. QUICK ACCESS TO RESULTS
print(f"\n🔗 Quick Access Variables:")
print(f"   final_chain_ctx = Complete results context")
print(f"   final_chain_ctx['decision']['action'] = '{final_chain_ctx.get('decision', {}).get('action', 'N/A')}'")
print(f"   final_chain_ctx['forecast']['model_used'] = '{final_chain_ctx.get('forecast', {}).get('model_used', 'N/A')}'")
print(f"   final_chain_ctx['sentiment']['verdict'] = '{final_chain_ctx.get('sentiment', {}).get('verdict', 'N/A')}'")

🚀 TESTING ENHANCED AGENTIC ORCHESTRATOR CHAIN
⏳ Executing enhanced pipeline...
✅ Latest Fetched Date: 2025-10-29
✅ Pipeline completed!

══════════════════════════════════════════════════
📈 FINAL TRADING DECISION (Enhanced Agentic Chain)
══════════════════════════════════════════════════
🔹 Action: HOLD
🔹 Rationale: Holding: confidence too low (conf 0.000 vs thresh 0.75); sentiment=neutral.

🔍 ENHANCED FORECAST DETAILS
🔹 Model Used: none (selected by composite score)
🔹 Predicted Movement: ERROR
🔹 Confidence Score: 0.0%

🧠 ENHANCED SENTIMENT ANALYSIS
🔹 Verdict: NEUTRAL (Weak Strength)
🔹 Score: 0.000 (range: -1 to +1)
📊 Sentiment Performance Metrics:
   • News Volume: 0 articles analyzed
   • News Freshness: 0.0% (recent news weight)
   • Historical Accuracy: 65.0% (directional hit rate)
   • Enhanced Confidence: 0.0% (volume + agreement + recency)
🔹 Forecast-Sentiment Alignment: ➖ NEUTRAL (no adjustment)
🔹 Used in Decision: ✅ YES

🛡️ EXPLAINABILITY TRACE
01. [OrchestratorChain] Starting p

In [ ]:
#Cell 8
import os
os.makedirs("tradiwin_core", exist_ok=True)

In [ ]:

#Cell 9
%%writefile tradiwin_core/__init__.py
# Re-export the public API so Streamlit can `from tradiwin_core import ...`
from .core import (
    OrchestratorChain,
    MCP_tradi_win_runner,
    build_default_context,
    load_prices,
    mini_backtest,
    mini_backtest_multi,
    diebold_mariano,
    compute_dm_table,  # NEW: convenience helper for DM comparison tables
)


Writing tradiwin_core/__init__.py


In [ ]:
#cell 10

%%writefile tradiwin_core/core.py
from __future__ import annotations
import os, warnings, random, math, json
from typing import Dict, Any, Tuple, List
import numpy as np
import pandas as pd
import yfinance as yf
from statsmodels.tsa.arima.model import ARIMA
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")

# -----------------------
# Context & Utilities
# -----------------------
def build_default_context(symbol: str = "BTC") -> Dict[str, Any]:
    return {
        "data": {"symbol": symbol, "df": None, "timestamp": None},
        "forecast": {
            "model_used": None, "predicted_movement": None, "confidence": 0.0,
            "metrics_per_model": {}, "skipped": [], "forecast_value": None, "latest_price": None,
            # NEW: allow the UI to ask ForecastAgent to include LSTM in selection
            "use_lstm_for_selection": False,
            # optional: backtest lookback to reuse in LSTM metric calc
            "selection_lookback_days": 30,
        },
        "sentiment": {
            "enabled": False, "source": None, "score": 0.0, "verdict": "neutral",
            "confidence": 0.0, "used_in_decision": False, "alignment": None,
            "sample_headlines": [], "strength_category": None, "news_volume": 0,
            "recency_score": 0.0, "directional_accuracy": 0.52
        },
        "decision": {"action": None, "rationale": None, "score": 0.0},
        "history": [],
        "sentiment_history": [],
        "explainability": {"enabled": True, "full_trace": []},
        "debug_info": {"enabled": False, "directional_accuracy_details": []}
    }

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = ["_".join([str(x) for x in tup if str(x) != ""]).strip()
                      for tup in df.columns.values]
    return df

def load_prices(symbol: str, days: int = 365) -> pd.DataFrame:
    df = yf.download(tickers=f"{symbol}-USD", period=f"{days}d", interval="1d", progress=False)
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["Date", "Close"])
    df = df.reset_index(drop=False)
    df = _flatten_columns(df)

    # normalize Date
    date_col = None
    for c in df.columns:
        if c == "Date":
            date_col = c; break
    if date_col is None:
        for c in df.columns:
            lc = c.lower()
            if lc in ("datetime","timestamp") or "date" in lc:
                date_col = c; break
    if date_col is None:
        first = df.columns[0]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            probe = pd.to_datetime(df[first], errors="coerce")
        if probe.notna().any():
            df.insert(0, "Date", probe)
            date_col = "Date"
    if date_col != "Date":
        df.rename(columns={date_col: "Date"}, inplace=True)
    if "Date" not in df.columns:
        return pd.DataFrame(columns=["Date", "Close"])

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df[df["Date"].notna()]
    df["Date"] = df["Date"].dt.tz_localize(None).dt.normalize()

    # normalize Close
    close_col = None
    for c in df.columns:
        if c.lower().startswith("close"):
            close_col = c; break
    if close_col is None:
        for c in df.columns:
            if "close" in c.lower():
                close_col = c; break
    if close_col is not None and close_col != "Close":
        df.rename(columns={close_col: "Close"}, inplace=True)
    if "Close" not in df.columns:
        return pd.DataFrame(columns=["Date", "Close"])

    df = df[df["Close"].notna()]
    df = df.sort_values("Date").drop_duplicates(subset=["Date"], keep="last")
    return df[["Date", "Close"]]

def compute_metrics(y_true, y_pred) -> Tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    rmse = float(np.sqrt(((y_true - y_pred) ** 2).mean()))
    mae  = float(np.abs(y_true - y_pred).mean())
    eps = 1e-8; den = np.maximum(np.abs(y_true), eps)
    mape = float(np.mean(np.abs((y_true - y_pred) / den)) * 100.0)
    return rmse, mae, mape

def calculate_directional_accuracy(y_true, y_pred, min_change_pct=0.1, debug_context=None) -> float:
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if len(y_true) < 2 or len(y_pred) < 2:
        return 0.0
    n = min(len(y_true), len(y_pred))
    y_true, y_pred = y_true[:n], y_pred[:n]
    if n < 2:
        return 0.0
    actual_changes = np.diff(y_true) / y_true[:-1] * 100
    predicted_changes = np.diff(y_pred) / y_pred[:-1] * 100
    significant_moves = np.abs(actual_changes) >= min_change_pct
    if not significant_moves.any():
        return 0.5
    a = np.sign(actual_changes[significant_moves])
    p = np.sign(predicted_changes[significant_moves])
    return float((a == p).sum() / len(a))

def composite_score(metrics: Dict[str, float], latest_price: float) -> float:
    try:
        normalized_mae = metrics["MAE"] / max(latest_price, 1e-8)
        dir_acc_score = metrics["Directional_Accuracy"]
        mape_score = max(0, 1 - metrics["MAPE"] / 100)
        mae_score = max(0, 1 - normalized_mae)
        return 0.4 * dir_acc_score + 0.3 * mape_score + 0.3 * mae_score
    except Exception:
        return 0.0

def _guard_ok(forecast_value: float, latest: float, rmse: float) -> bool:
    if not (np.isfinite(forecast_value) and np.isfinite(latest) and np.isfinite(rmse)): return False
    if latest <= 0: return False
    if rmse / latest > 0.80: return False
    if abs(forecast_value - latest) / latest > 0.50: return False
    return True

def _news_query(symbol: str) -> str:
    return {"BTC":"bitcoin","ETH":"ethereum","SOL":"solana","XRP":"ripple xrp","ADA":"cardano","BNB":"binance coin"}.get(symbol.upper(), symbol)

# -----------------------
# Mini backtests
# -----------------------
def mini_backtest(close_df: pd.DataFrame, asof_ts: pd.Timestamp, lookback_days: int = 30) -> pd.DataFrame:
    df = close_df.copy()
    if df.empty: return pd.DataFrame()
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None).dt.normalize()
    asof_ts = pd.Timestamp(asof_ts).normalize()
    df = df[df["Date"] <= asof_ts].copy()
    if len(df) < 60: return pd.DataFrame()

    tail = df.tail(lookback_days + 1).reset_index(drop=True)
    rows = []
    all_predictions, all_actuals = [], []

    for i in range(len(tail) - 1):
        cut_date = pd.Timestamp(tail["Date"].iloc[i]).normalize()
        hist = df[df["Date"] <= cut_date]["Close"].to_numpy()
        try:
            f = float(ARIMA(hist, order=(3,1,0)).fit().forecast()[0])
        except Exception:
            f = float(hist[-1])
        actual_next = float(tail.loc[i+1, "Close"])
        all_predictions.append(f); all_actuals.append(actual_next)

    overall_dir_acc = calculate_directional_accuracy(all_actuals, all_predictions, min_change_pct=0.1)

    for i in range(len(tail) - 1):
        cut_date = pd.Timestamp(tail["Date"].iloc[i]).normalize()
        hist = df[df["Date"] <= cut_date]["Close"].to_numpy()
        try:
            f = float(ARIMA(hist, order=(3,1,0)).fit().forecast()[0])
        except Exception:
            f = float(hist[-1])

        next_day = pd.Timestamp(tail["Date"].iloc[i+1]).normalize()
        actual_next = float(tail.loc[i+1, "Close"])
        latest = float(hist[-1])
        price_error_pct = abs(f - actual_next) / max(actual_next, 1e-8)
        rmse_conf = max(0.0, min(1.0, 1 - price_error_pct))
        confidence = 0.6 * overall_dir_acc + 0.4 * rmse_conf
        pred_dir = "UP" if f > latest else "DOWN" if f < latest else "STABLE"
        actual_dir = "UP" if actual_next > latest else "DOWN" if actual_next < latest else "STABLE"
        dir_correct = (pred_dir == actual_dir)
        rows.append({
            "date": next_day.date(),
            "predicted_close": f,
            "actual_close": actual_next,
            "pred_direction": pred_dir,
            "actual_direction": actual_dir,
            "directional_correct": dir_correct,
            "conf": confidence
        })
    return pd.DataFrame(rows)

def _pred_naive(hist: np.ndarray) -> float:
    return float(hist[-1])

def _pred_ses(hist: np.ndarray) -> float:
    try:
        from statsmodels.tsa.holtwinters import SimpleExpSmoothing
        model = SimpleExpSmoothing(hist, initialization_method="heuristic").fit(optimized=True)
        return float(model.forecast(1)[0])
    except Exception:
        return float(hist[-1])

def _pred_arima(hist: np.ndarray) -> float:
    try:
        return float(ARIMA(hist, order=(3,1,0)).fit().forecast()[0])
    except Exception:
        return float(hist[-1])

def _pred_prophet(dates: np.ndarray, values: np.ndarray) -> float:
    try:
        from prophet import Prophet
        d = pd.DataFrame({"ds": pd.to_datetime(dates), "y": values})
        m = Prophet(weekly_seasonality=False, yearly_seasonality=False) if len(d) < 180 else Prophet()
        m.fit(d)
        fut = m.make_future_dataframe(periods=1, freq="D")
        fc = m.predict(fut)
        return float(fc["yhat"].iloc[-1])
    except Exception:
        return float(values[-1])

def _lstm_next_price(hist_vals: np.ndarray, epochs: int = 10) -> float:
    """Train on returns and predict next price from last close."""
    try:
        import tensorflow as tf
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import LSTM, Dense, Dropout
        from sklearn.preprocessing import MinMaxScaler
        np.random.seed(42); random.seed(42)
        try: tf.random.set_seed(42)
        except Exception: pass

        if len(hist_vals) < 30:
            return float(hist_vals[-1])

        returns = np.diff(hist_vals) / hist_vals[:-1] * 100
        sc = MinMaxScaler(feature_range=(-1, 1))
        scaled = sc.fit_transform(returns.reshape(-1, 1)).flatten()
        lookback = min(12, len(scaled) - 1)
        if lookback < 2:
            return float(hist_vals[-1])

        X, y = [], []
        for j in range(len(scaled) - lookback):
            X.append(scaled[j:j+lookback]); y.append(scaled[j+lookback])
        X, y = np.array(X), np.array(y)
        if len(X) == 0:
            return float(hist_vals[-1])

        model = Sequential([
            tf.keras.layers.Input(shape=(lookback, 1)),
            LSTM(50, return_sequences=True),
            Dropout(0.2),
            LSTM(25, return_sequences=False),
            Dropout(0.2),
            Dense(1)
        ])
        model.compile(optimizer="adam", loss="mse")
        model.fit(X.reshape(X.shape[0], X.shape[1], 1), y, epochs=epochs, verbose=0, validation_split=0.2)
        last_seq = scaled[-lookback:].reshape(1, lookback, 1)
        pred_ret_scaled = model.predict(last_seq, verbose=0)[0][0]
        pred_ret = sc.inverse_transform([[pred_ret_scaled]])[0][0]
        return float(hist_vals[-1] * (1 + pred_ret / 100))
    except Exception:
        return float(hist_vals[-1])

def mini_backtest_multi(
    close_df: pd.DataFrame,
    asof_ts: pd.Timestamp,
    lookback_days: int = 30,
    models: List[str] | None = None,
    include_lstm: bool = False,
    lstm_epochs: int = 10
) -> pd.DataFrame:
    if models is None:
        models = ["Naive", "SES", "ARIMA", "Prophet"]

    df = close_df.copy()
    if df.empty: return pd.DataFrame()
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None).dt.normalize()
    asof_ts = pd.Timestamp(asof_ts).normalize()
    df = df[df["Date"] <= asof_ts].copy()
    if len(df) < 120:
        return pd.DataFrame()

    tail = df.tail(lookback_days + 1).reset_index(drop=True)
    preds = {m: [] for m in models}
    dates, actuals = [], []

    for i in range(len(tail) - 1):
        cut_date = pd.Timestamp(tail["Date"].iloc[i]).normalize()
        hist_df = df[df["Date"] <= cut_date].copy()
        hist_vals = hist_df["Close"].to_numpy(dtype=float)
        hist_dates = hist_df["Date"].to_numpy()

        if "Naive" in models:   preds["Naive"].append(float(_pred_naive(hist_vals)))
        if "SES" in models:     preds["SES"].append(float(_pred_ses(hist_vals)))
        if "ARIMA" in models:   preds["ARIMA"].append(float(_pred_arima(hist_vals)))
        if "Prophet" in models: preds["Prophet"].append(float(_pred_prophet(hist_dates, hist_vals)))

        if include_lstm and "LSTM" in models:
            preds.setdefault("LSTM", []).append(_lstm_next_price(hist_vals, epochs=lstm_epochs))

        next_day = pd.Timestamp(tail["Date"].iloc[i+1]).normalize()
        actual_next = float(tail.loc[i+1, "Close"])
        dates.append(next_day.date()); actuals.append(actual_next)

    out = pd.DataFrame({"date": dates, "actual_close": actuals})
    for m in models:
        ser = preds.get(m, [])
        if len(ser) == len(dates):
            out[f"predicted_{m}"] = np.asarray(ser, dtype=float)
    return out

# -----------------------
# DM test + table
# -----------------------
def diebold_mariano(e1: np.ndarray, e2: np.ndarray, h: int = 1, power: int = 2) -> Tuple[float, float]:
    e1 = np.asarray(e1, dtype=float).reshape(-1)
    e2 = np.asarray(e2, dtype=float).reshape(-1)
    n = min(len(e1), len(e2))
    e1, e2 = e1[:n], e2[:n]
    if n < 5: return (np.nan, np.nan)
    d = (np.abs(e1) ** power) - (np.abs(e2) ** power) if power != 1 else (np.abs(e1) - np.abs(e2))
    dbar = d.mean()
    s2 = np.var(d, ddof=1) if h <= 1 else np.var(d, ddof=1)
    if s2 <= 0 or not np.isfinite(s2): return (np.nan, np.nan)
    from math import erfc, sqrt
    dm = dbar / np.sqrt(s2 / n); p = erfc(abs(dm) / sqrt(2.0))
    return (float(dm), float(p))

def compute_dm_table(backtest_df: pd.DataFrame, selected_model: str) -> pd.DataFrame:
    if backtest_df is None or backtest_df.empty: return pd.DataFrame()
    pred_cols = [c for c in backtest_df.columns if c.startswith("predicted_")]
    if not pred_cols: return pd.DataFrame()
    actual = backtest_df["actual_close"].values.astype(float)
    sel_col = f"predicted_{selected_model}"
    if sel_col not in pred_cols: return pd.DataFrame()

    e_sel = (actual - backtest_df[sel_col].values.astype(float))
    rows = []
    for c in pred_cols:
        if c == sel_col: continue
        name = c.replace("predicted_","")
        e_alt = (actual - backtest_df[c].values.astype(float))
        dm, p = diebold_mariano(e_alt, e_sel, h=1, power=2)
        winner = "—" if np.isnan(p) else ("Alt better" if (p < 0.10 and dm < 0) else ("Selected better" if p < 0.10 else "No significant diff"))
        rows.append({"Model": name, "DM_stat": None if np.isnan(dm) else float(dm), "p_value": None if np.isnan(p) else float(p), "Winner": winner})
    return pd.DataFrame(rows)

# -----------------------
# Agents
# -----------------------
class AgentBase:
    def run(self, context: Dict[str, Any]) -> Dict[str, Any]:
        raise NotImplementedError

class DataFetcherAgent(AgentBase):
    def __init__(self, days: int = 365): self.days = days
    def run(self, context: Dict[str, Any]) -> Dict[str, Any]:
        symbol = context["data"]["symbol"]
        df = load_prices(symbol, days=self.days)
        asof_str = context["data"].get("timestamp")
        if asof_str:
            asof_ts = pd.to_datetime(asof_str).tz_localize(None).normalize()
            df = df[df["Date"] <= asof_ts].copy()
        if df.empty:
            context["explainability"]["full_trace"].append("[Data] No price data.")
            context["data"]["df"] = pd.DataFrame(columns=["Date","Close"])
            context["data"]["timestamp"] = None
            return context
        context["data"]["df"] = df
        context["data"]["timestamp"] = str(df["Date"].iloc[-1].date())
        context["explainability"]["full_trace"].append(
            f"[Data] Loaded {len(df)} rows | range {df['Date'].min().date()} → {df['Date'].max().date()}"
        )
        return context

class ForecastAgent(AgentBase):
    def run(self, context: Dict[str, Any]) -> Dict[str, Any]:
        df = context["data"].get("df")
        if df is None or df.empty:
            context["explainability"]["full_trace"].append("[Forecast] No data.")
            return context
        prices = df["Close"].to_numpy(dtype=float)
        if len(prices) < 30:
            context["explainability"]["full_trace"].append("[Forecast] Not enough history (<30).")
            return context
        latest = float(prices[-1])
        results, metrics, skipped = {}, {}, []

        # ARIMA
        try:
            arima = ARIMA(prices, order=(3,1,0)).fit()
            f = float(arima.forecast()[0]); pred = arima.predict()
            validation_size = min(35, len(pred))
            rmse, mae, mape = compute_metrics(prices[-validation_size:], pred[-validation_size:])
            dir_acc = calculate_directional_accuracy(prices[-validation_size:], pred[-validation_size:], 0.1)
            if _guard_ok(f, latest, rmse):
                results["ARIMA"] = f
                metrics["ARIMA"] = {"RMSE": rmse, "MAE": mae, "MAPE": mape, "Directional_Accuracy": dir_acc}
            else: skipped.append(("ARIMA", f"guardrails (RMSE={rmse:.0f})"))
        except Exception as e:
            skipped.append(("ARIMA", f"error: {str(e).splitlines()[0][:120]}"))

        # Prophet
        try:
            from prophet import Prophet
            d = df.rename(columns={"Date":"ds","Close":"y"})[["ds","y"]].copy()
            m = Prophet(weekly_seasonality=False, yearly_seasonality=False) if len(d) < 180 else Prophet()
            m.fit(d)
            fut = m.make_future_dataframe(periods=1, freq="D")
            fc = m.predict(fut); f = float(fc["yhat"].iloc[-1])
            validation_size = min(35, len(fc) - 1)
            y_true = d["y"].iloc[-validation_size:].to_numpy()
            y_pred = fc["yhat"].iloc[-(validation_size+1):-1].to_numpy()
            rmse, mae, mape = compute_metrics(y_true, y_pred)
            dir_acc = calculate_directional_accuracy(y_true, y_pred, 0.1)
            if _guard_ok(f, latest, rmse):
                results["Prophet"] = f
                metrics["Prophet"] = {"RMSE": rmse, "MAE": mae, "MAPE": mape, "Directional_Accuracy": dir_acc}
            else: skipped.append(("Prophet", f"guardrails (RMSE={rmse:.0f})"))
        except Exception as e:
            skipped.append(("Prophet", f"error: {str(e).splitlines()[0][:120]}"))

        # Baselines
        try:
            naive_pred = _pred_naive(prices)
            naive_hist_pred = np.r_[prices[:-1]]
            rmse, mae, mape = compute_metrics(prices[1:], naive_hist_pred)
            dir_acc = calculate_directional_accuracy(prices[1:], naive_hist_pred, 0.1)
            results["Naive"] = naive_pred
            metrics["Naive"] = {"RMSE": rmse, "MAE": mae, "MAPE": mape, "Directional_Accuracy": dir_acc}
        except Exception: pass
        try:
            ses_pred = _pred_ses(prices)
            from statsmodels.tsa.holtwinters import SimpleExpSmoothing
            ses = SimpleExpSmoothing(prices, initialization_method="heuristic").fit(optimized=True)
            ses_hist_pred = np.r_[np.nan, ses.fittedvalues[:-1]]
            valid = ~np.isnan(ses_hist_pred)
            rmse, mae, mape = compute_metrics(prices[valid], ses_hist_pred[valid])
            dir_acc = calculate_directional_accuracy(prices[valid], ses_hist_pred[valid], 0.1)
            results["SES"] = ses_pred
            metrics["SES"] = {"RMSE": rmse, "MAE": mae, "MAPE": mape, "Directional_Accuracy": dir_acc}
        except Exception: pass

        # OPTIONAL: LSTM (competes for selection if asked)
        if context["forecast"].get("use_lstm_for_selection", False):
            try:
                f_lstm = _lstm_next_price(prices, epochs=10)
                # obtain metrics from a tiny rolling backtest for fairness
                lookback = int(context["forecast"].get("selection_lookback_days", 30))
                asof_str = context["data"].get("timestamp")
                asof_ts = pd.to_datetime(asof_str).tz_localize(None).normalize() if asof_str else df["Date"].iloc[-1]
                bt = mini_backtest_multi(
                    close_df=df[["Date","Close"]].rename(columns={"Date":"Date","Close":"Close"}),
                    asof_ts=asof_ts,
                    lookback_days=lookback,
                    models=["LSTM"],
                    include_lstm=True,
                    lstm_epochs=10
                )
                if not bt.empty and "predicted_LSTM" in bt.columns:
                    y_true = bt["actual_close"].astype(float).values
                    y_pred = bt["predicted_LSTM"].astype(float).values
                    rmse, mae, mape = compute_metrics(y_true, y_pred)
                    # dir acc
                    dir_acc = calculate_directional_accuracy(y_true, y_pred, 0.1)
                else:
                    # fallback: weak metrics if backtest unavailable
                    rmse, mae, mape, dir_acc = latest*0.05, latest*0.03, 5.0, 0.52

                if _guard_ok(f_lstm, latest, rmse):
                    results["LSTM"] = float(f_lstm)
                    metrics["LSTM"] = {"RMSE": float(rmse), "MAE": float(mae), "MAPE": float(mape), "Directional_Accuracy": float(dir_acc)}
                else:
                    skipped.append(("LSTM", f"guardrails (RMSE={rmse:.0f})"))
            except Exception as e:
                skipped.append(("LSTM", f"error: {str(e).splitlines()[0][:120]}"))

        if not metrics:
            context["forecast"].update({
                "model_used":"none","predicted_movement":"stable","confidence":0.0,
                "metrics_per_model":{}, "skipped": skipped,
                "forecast_value": latest, "latest_price": latest
            })
            context["explainability"]["full_trace"].append("[Forecast] All models failed/skipped.")
            return context

        model_scores = {m: composite_score(mt, latest) for m, mt in metrics.items()}
        best = max(model_scores.items(), key=lambda x: x[1])[0]
        fval = results[best]; best_metrics = metrics[best]
        directional_conf = best_metrics["Directional_Accuracy"]
        rmse_conf = max(0.0, min(1.0, 1 - (best_metrics["RMSE"] / max(latest,1e-8))))
        conf = float(0.6 * directional_conf + 0.4 * rmse_conf)
        move = "up" if fval>latest else "down" if fval<latest else "stable"

        context["forecast"].update({
            "model_used": best, "predicted_movement": move, "confidence": round(conf,3),
            "metrics_per_model": metrics, "skipped": skipped,
            "forecast_value": fval, "latest_price": latest
        })
        context["explainability"]["full_trace"].append(
            f"[Forecast] {best} → {move.upper()} (conf={conf:.3f}) | "
            f"Dir_Acc={best_metrics['Directional_Accuracy']:.1%}, RMSE=${best_metrics['RMSE']:,.2f}"
        )
        return context

class SentimentAgent(AgentBase):
    def _proxy_from_prices(self, df: pd.DataFrame) -> Dict[str, Any]:
        if df is None or df.empty or len(df) < 10:
            return {"score": 0.0, "verdict": "neutral", "confidence": 0.33,
                    "strength_category": "Weak", "directional_accuracy": 0.52}
        px = df["Close"].astype(float).values
        ret7 = (px[-1] / px[max(0, len(px)-8)] - 1.0) if len(px) >= 8 else 0.0
        score = float(np.tanh(ret7 * 3))
        verdict = "bullish" if score > 0.20 else "bearish" if score < -0.20 else "neutral"
        strength = "Strong" if abs(score) >= 0.6 else ("Moderate" if abs(score) >= 0.2 else "Weak")
        confidence = float(0.5 * abs(score) + 0.2)
        return {"score": score, "verdict": verdict, "confidence": confidence,
                "strength_category": strength, "directional_accuracy": 0.52}

    def _fetch_headlines(self, symbol: str) -> List[Dict[str, str]]:
        api_key = os.environ.get("NEWS_API_KEY", "")
        q = {"BTC":"bitcoin","ETH":"ethereum","SOL":"solana","XRP":"ripple xrp","ADA":"cardano","BNB":"binance coin"}.get(symbol.upper(), symbol)
        items: List[Dict[str, str]] = []
        if not api_key:
            now = datetime.utcnow().date().isoformat()
            return [
                {"title": f"{q.title()} on-chain flows steady; market eyes macro data ({now})", "source": "DemoWire", "url": ""},
                {"title": f"Analysts debate {q} valuation after recent rally", "source": "DemoWire", "url": ""},
                {"title": f"{q.upper()} developers ship minor upgrade; fees stable", "source": "DemoWire", "url": ""},
            ]
        try:
            import requests
            url = "https://newsapi.org/v2/everything"
            params = {"apiKey": api_key, "q": q, "pageSize": 10, "sortBy": "publishedAt", "language": "en"}
            r = requests.get(url, params=params, timeout=8)
            if r.status_code == 200:
                data = r.json()
                for a in (data.get("articles") or [])[:10]:
                    items.append({
                        "title": a.get("title") or "",
                        "source": (a.get("source") or {}).get("name") or "",
                        "url": a.get("url") or ""
                    })
        except Exception:
            pass
        if not items:
            items = [{"title": f"{q.title()} sees mixed flows as traders stay cautious",
                      "source": "DemoWire", "url": ""}]
        return items

    def run(self, context: Dict[str, Any]) -> Dict[str, Any]:
        if not context.get("sentiment", {}).get("enabled", False):
            context["explainability"]["full_trace"].append("[Sentiment] Disabled; skipping.")
            return context
        symbol = context["data"]["symbol"]
        df = context["data"].get("df")
        proxy = self._proxy_from_prices(df)
        headlines = self._fetch_headlines(symbol)
        news_volume = len(headlines)
        recency_score = 0.7 if news_volume >= 5 else (0.4 if news_volume >= 2 else 0.2)

        context["sentiment"].update({
            "source": "Proxy/NewsHybrid",
            "score": float(proxy["score"]),
            "verdict": proxy["verdict"],
            "confidence": float(proxy["confidence"]),
            "strength_category": proxy["strength_category"],
            "news_volume": news_volume,
            "recency_score": recency_score,
            "directional_accuracy": float(proxy["directional_accuracy"]),
            "sample_headlines": headlines
        })
        context["explainability"]["full_trace"].append(
            f"[Sentiment] Proxy score={proxy['score']:+.3f} ({proxy['verdict']}, {proxy['strength_category']}); headlines={news_volume}"
        )
        return context

class DecisionAgent(AgentBase):
    def run(self, context: Dict[str, Any], base_threshold: float = 0.70) -> Dict[str, Any]:
        fc, st = context["forecast"], context["sentiment"]
        move = fc.get("predicted_movement","stable"); conf = float(fc.get("confidence",0.0))
        sv = st.get("verdict","neutral") if st.get("enabled") else "neutral"

        if move == "up":
            aligned = (sv == "bullish")
            thr = base_threshold - 0.10 if aligned else (base_threshold + 0.10 if sv == "bearish" else base_threshold)
        elif move == "down":
            aligned = (sv == "bearish")
            thr = base_threshold - 0.10 if aligned else (base_threshold + 0.10 if sv == "bullish" else base_threshold)
        else:
            aligned = None; thr = max(0.75, base_threshold)
        if sv == "neutral": aligned = None

        if move == "up" and conf >= thr:
            action, rationale = "buy", f"Forecast {move.upper()} with confidence {conf:.3f} (threshold {thr:.2f}); sentiment={sv}."
        elif move == "down" and conf >= thr:
            action, rationale = "sell", f"Forecast {move.upper()} with confidence {conf:.3f} (threshold {thr:.2f}); sentiment={sv}."
        else:
            reason = "confidence too low" if conf < thr else "no directional edge"
            action, rationale = "hold", f"Holding: {reason} (conf {conf:.3f} vs thresh {thr:.2f}); sentiment={sv}."

        context["decision"].update({"action": action, "rationale": rationale, "score": round(conf,3)})
        context["sentiment"]["used_in_decision"] = bool(st.get("enabled"))
        context["sentiment"]["alignment"] = aligned
        context["history"].append({
            "timestamp": context["data"].get("timestamp"),
            "forecast_movement": move, "forecast_confidence": conf,
            "sentiment_verdict": sv, "sentiment_score": st.get("score"),
            "sentiment_confidence": st.get("confidence"),
            "sentiment_strength": st.get("strength_category"),
            "news_volume": st.get("news_volume"),
            "recency_score": st.get("recency_score"),
            "alignment": aligned, "used_sentiment": st.get("enabled", False)
        })
        context["explainability"]["full_trace"].append(f"[Decision] {action.upper()} → {rationale}")
        return context

class OrchestratorChain:
    def __init__(self):
        self.data_fetcher = DataFetcherAgent()
        self.forecaster   = ForecastAgent()
        self.sentimenter  = SentimentAgent()
        self.decider      = DecisionAgent()

    def run(self, context: Dict[str, Any], base_threshold: float = 0.70) -> Dict[str, Any]:
        try: context = self.data_fetcher.run(context);   context["explainability"]["full_trace"].append("[Chain] DataFetcher completed")
        except Exception as e: context["explainability"]["full_trace"].append(f"[Chain] DataFetcher error: {e}")
        try: context = self.forecaster.run(context);     context["explainability"]["full_trace"].append("[Chain] ForecastAgent completed")
        except Exception as e: context["explainability"]["full_trace"].append(f"[Chain] ForecastAgent error: {e}")
        try: context = self.sentimenter.run(context);    context["explainability"]["full_trace"].append("[Chain] SentimentAgent completed")
        except Exception as e: context["explainability"]["full_trace"].append(f"[Chain] SentimentAgent error: {e}")
        try: context = self.decider.run(context, base_threshold=base_threshold); context["explainability"]["full_trace"].append("[Chain] DecisionAgent completed")
        except Exception as e: context["explainability"]["full_trace"].append(f"[Chain] DecisionAgent error: {e}")
        return context

def MCP_tradi_win_runner(context: Dict[str, Any], use_sentiment: bool = True, base_threshold: float = 0.70) -> Dict[str, Any]:
    context["sentiment"]["enabled"] = use_sentiment
    return OrchestratorChain().run(context, base_threshold=base_threshold)


Writing tradiwin_core/core.py


In [ ]:
#cell 11

%%writefile streamlit_app.py
# --- TRADI-WINNING • Fast, Explainable Crypto Forecasting (Updated with LSTM selection toggle) ---

import os, json, time, re, numpy as np, pandas as pd, streamlit as st, altair as alt
from datetime import datetime
import importlib, sys
from typing import Dict, Any, List

def fmt_pct(x, places=1):
    try:
        return f"{float(x)*100:.{places}f}%"
    except Exception:
        return "—"

def fmt_sig(x, sig=2):
    try:
        x = float(x)
        if x == 0:
            return "0"
        from math import log10, floor
        p = -int(floor(log10(abs(x)))) + (sig - 1)
        p = max(p, 0)
        return f"{x:.{p}f}"
    except Exception:
        return "—"

def to_float(x, default: float = 0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default

def to_int(x, default: int = 0) -> int:
    try:
        return int(float(x))
    except Exception:
        return default

def verdict_color(verdict: str) -> str:
    v = (verdict or "").lower()
    if v == "bullish": return "#10B981"
    if v == "bearish": return "#EF4444"
    return "#6B7280"

def badge(text: str, color: str):
    return f"<span style='display:inline-block;padding:2px 8px;border-radius:10px;background:{color}20;color:{color};font-size:0.85rem'>{text}</span>"

def pretty_model(name: str) -> str:
    return {"Naive":"Naive (Random Walk)","SES":"Simple Exponential Smoothing (SES)","ARIMA":"ARIMA","Prophet":"Prophet","LSTM":"LSTM"}.get(name, name)

# ---- import core
try:
    if "tradiwin_core" in sys.modules:
        import tradiwin_core
        importlib.reload(tradiwin_core)
    else:
        import tradiwin_core
    from tradiwin_core import OrchestratorChain, MCP_tradi_win_runner, build_default_context, load_prices, mini_backtest_multi, compute_dm_table
    core_ok = True
except Exception as e:
    st.error(f"Core import failed: {e}")
    st.stop()

st.set_page_config(page_title="TRADI-WINNING", layout="wide", page_icon="📈", initial_sidebar_state="expanded")
ART_DIR = "artifacts"; os.makedirs(ART_DIR, exist_ok=True)

@st.cache_data(show_spinner=False)
def cached_prices(symbol: str, days: int = 365):
    return load_prices(symbol, days)

@st.cache_data(show_spinner=False)
def cached_backtest(df_full: pd.DataFrame, asof_ts: pd.Timestamp, lookback_days: int, models: list, include_lstm: bool):
    return mini_backtest_multi(df_full, asof_ts, lookback_days=lookback_days, models=models, include_lstm=include_lstm)

@st.cache_resource(show_spinner=False)
def _get_finbert():
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
        model_name = "ProsusAI/finbert"
        tok = AutoTokenizer.from_pretrained(model_name)
        mdl = AutoModelForSequenceClassification.from_pretrained(model_name)
        nlp = pipeline("text-classification", model=mdl, tokenizer=tok, return_all_scores=True)
        return nlp
    except Exception:
        return None

def run_finbert_on_headlines(headlines: List[str]) -> Dict[str, Any]:
    nlp = _get_finbert()
    if nlp is None or not headlines:
        return {}
    scores = []
    for t in headlines:
        out = nlp(t[:512])[0]
        row = {d["label"].lower(): float(d["score"]) for d in out}
        fin_score = row.get("positive", 0.0) - row.get("negative", 0.0)
        scores.append({"text": t,"positive": row.get("positive", 0.0),"neutral": row.get("neutral", 0.0),"negative": row.get("negative", 0.0),"finbert_score": fin_score})
    if not scores:
        return {}
    avg_pos = float(np.mean([s["positive"] for s in scores]))
    avg_neg = float(np.mean([s["negative"] for s in scores]))
    avg_neu = float(np.mean([s["neutral"] for s in scores]))
    avg_fin = float(np.mean([s["finbert_score"] for s in scores]))
    verdict = "bullish" if avg_fin > 0.1 else "bearish" if avg_fin < -0.1 else "neutral"
    conf = float(max(abs(avg_fin), max(avg_pos, avg_neg)))
    return {"avg_positive": avg_pos,"avg_neutral": avg_neu,"avg_negative": avg_neg,"avg_score": avg_fin,"verdict": verdict,"confidence": conf}

@st.cache_data(show_spinner=False)
def cached_llm_explain(openai_key: str, payload: dict) -> str:
    if not openai_key:
        return "LLM explanation unavailable (no OPENAI_API_KEY)."
    try:
        from openai import OpenAI
        client = OpenAI(api_key=openai_key)
        prompt = f"""You are an explainable-AI narrator for a crypto forecasting dashboard.
Write 9–12 sentences. Use percentages for all confidences & composite scores.
Composite picks the winning model (40% DirAcc + 30% (1−MAPE) + 30% (1−MAE/price)).
Forecast Confidence (decision metric) = 0.6×DirAcc + 0.4×(1−RMSE/price).

State the analysis date, the selected model and its composite vs peers, today’s confidence vs (adjusted) threshold with the action math,
sentiment (and any threshold shift), any DM p<0.10 result, and a short caution.

JSON:
{json.dumps(payload, indent=2, default=str)}
"""
        resp = client.chat.completions.create(model="gpt-4.1", messages=[{"role":"user","content":prompt}], temperature=0.25)
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"LLM explanation error: {e}"

# Sidebar
with st.sidebar:
    st.markdown("## 🎛️ Trading Controls")
    CRYPTO_MAP = {"Bitcoin (BTC)":"BTC","Ethereum (ETH)":"ETH","Solana (SOL)":"SOL","Ripple (XRP)":"XRP","Cardano (ADA)":"ADA","BNB Chain (BNB)":"BNB","Polygon (MATIC)":"MATIC","Polkadot (DOT)":"DOT","Chainlink (LINK)":"LINK","Avalanche (AVAX)":"AVAX","Uniswap (UNI)":"UNI","Litecoin (LTC)":"LTC","Cosmos (ATOM)":"ATOM","Fantom (FTM)":"FTM","Algorand (ALGO)":"ALGO"}
    selected_crypto = st.selectbox("Select Cryptocurrency", list(CRYPTO_MAP.keys()))
    symbol = CRYPTO_MAP[selected_crypto]
    crypto_full_name = selected_crypto.split(" (")[0]

    df_full = cached_prices(symbol)
    if df_full.empty:
        st.error("❌ No price data found. Try a different cryptocurrency.")
        st.stop()

    min_d, max_d = df_full["Date"].min(), df_full["Date"].max()
    asof = st.date_input("Analysis Date", value=max_d.date(), min_value=min_d.date(), max_value=max_d.date())
    asof_ts = pd.to_datetime(asof).tz_localize(None).normalize()

    st.markdown("### ⚙️ Advanced")
    base_thresh_pct = st.slider("Decision Threshold (%)", 50, 90, 70, 1, key="base_thresh_pct")
    st.session_state["base_threshold"] = base_thresh_pct / 100.0

    lookback_days = st.slider("Backtest window (days)", 10, 45, 30, 1)

    include_lstm_backtest = st.checkbox("Include LSTM (backtest + live table)", value=False)
    allow_lstm_selection = st.checkbox("Let LSTM compete for selection", value=False)

    enable_finbert = st.checkbox("Use FinBERT deep sentiment (slower)", value=False)

    run = st.button("🚀 Run Enhanced Analysis", use_container_width=True)

st.markdown(f"### TRADI-WINNING — {crypto_full_name} ({symbol})\n**Date:** {asof_ts.date()} • **Threshold:** {base_thresh_pct}%")

if run:
    t0 = time.time()
    ctx = build_default_context(symbol=symbol)
    ctx["sentiment"]["enabled"] = True
    ctx["data"]["timestamp"] = str(asof_ts.date())
    # pass LSTM selection flag + lookback for its metric calc
    ctx["forecast"]["use_lstm_for_selection"] = bool(allow_lstm_selection)
    ctx["forecast"]["selection_lookback_days"] = int(lookback_days)

    chain = OrchestratorChain()
    ctx = chain.run(ctx, base_threshold=st.session_state.get("base_threshold", 0.70))

    forecast = ctx.get("forecast", {})
    decision = ctx.get("decision", {})
    sentiment = ctx.get("sentiment", {})
    if not decision.get("score") or decision.get("score") <= 0:
        decision["score"] = float(forecast.get("confidence", 0.0))

    models = ["Naive", "SES", "ARIMA", "Prophet"] + (["LSTM"] if include_lstm_backtest else [])
    bt = cached_backtest(df_full, asof_ts, lookback_days, models=models, include_lstm=include_lstm_backtest)

    selected_model = forecast.get("model_used") or "ARIMA"
    dm_tbl = compute_dm_table(bt, selected_model=selected_model) if (bt is not None and not bt.empty) else pd.DataFrame()

    finbert_out = {}
    headline_texts = [h.get("title") for h in sentiment.get("sample_headlines", []) if isinstance(h, dict) and h.get("title")]
    if enable_finbert and headline_texts:
        finbert_out = run_finbert_on_headlines(headline_texts)

    # composite per model for LLM
    composite_per_model = {}
    latest_price = forecast.get("latest_price", 1.0) or 1.0
    for name, met in (forecast.get("metrics_per_model") or {}).items():
        dir_acc = to_float(met.get("Directional_Accuracy", 0))
        mape = to_float(met.get("MAPE", 0))
        mae = to_float(met.get("MAE", 0))
        comp = 0.4 * dir_acc + 0.3 * max(0, 1 - mape / 100) + 0.3 * max(0, 1 - (mae / max(latest_price, 1e-8)))
        composite_per_model[name] = comp

    openai_key = os.environ.get("OPENAI_API_KEY", "")
    payload = {
        "symbol": symbol,
        "date": str(asof_ts.date()),
        "threshold": float(st.session_state.get("base_threshold", 0.70)),
        "forecast": forecast,
        "decision": decision,
        "sentiment": sentiment,
        "finbert": finbert_out,
        "selected_model": selected_model,
        "dm_summary": [
            r for r in (dm_tbl.to_dict("records") if (dm_tbl is not None and not dm_tbl.empty) else [])
            if r.get("p_value") is not None and r["p_value"] < 0.10
        ],
        "composite_per_model": composite_per_model
    }
    llm_text = cached_llm_explain(openai_key, payload)

    st.session_state["ctx"] = ctx
    st.session_state["bt"] = bt
    st.session_state["dm"] = dm_tbl
    st.session_state["finbert"] = finbert_out
    st.session_state["llm_payload"] = payload
    st.session_state["llm_text"] = llm_text
    st.session_state["timings"] = {"total_sec": round(time.time() - t0, 2)}

    for name, data in {"decision": decision, "forecast": forecast, "sentiment": sentiment,
                       "dm_tests": dm_tbl.to_dict("records") if not dm_tbl.empty else [],
                       "finbert": finbert_out, "llm_explanation": llm_text}.items():
        with open(os.path.join(ART_DIR, f"{name}.json"), "w") as f:
            json.dump(data, f, indent=2, default=str)

tab_overview, tab_forecast, tab_sentiment, tab_corr, tab_explain = st.tabs(
    ["🎯 Overview", "📊 Forecasts", "🧠 Sentiment", "📈 Correlation", "🛡️ Explainability"]
)

ctx = st.session_state.get("ctx")
bt = st.session_state.get("bt")
dm_tbl = st.session_state.get("dm")
finbert_out = st.session_state.get("finbert", {})
llm_text = st.session_state.get("llm_text", "Run the analysis to generate explanation.")
llm_payload = st.session_state.get("llm_payload", {})

with tab_overview:
    st.subheader("📋 Trading Decision Overview")
    if not ctx:
        st.info("Run the analysis to populate this view.")
    else:
        forecast = ctx["forecast"]; decision = ctx["decision"]; sentiment = ctx["sentiment"]
        c1,c2,c3,c4 = st.columns(4)
        with c1:
            action = (decision.get("action") or "—").upper()
            clr = "#10B981" if action=="BUY" else "#EF4444" if action=="SELL" else "#F59E0B"
            st.markdown(f"**Action**  \n<span style='font-size:2rem;color:{clr};'>{action}</span>", unsafe_allow_html=True)
        with c2:
            dec_score = to_float(decision.get("score", forecast.get("confidence", 0.0)), 0.0)
            st.markdown(f"**Decision Confidence**  \n<span style='font-size:2rem;'>{fmt_pct(dec_score,1)}</span>", unsafe_allow_html=True)
        with c3:
            st.markdown(f"**Selected Model**  \n<span style='font-size:2rem;'>{forecast.get('model_used','—')}</span>", unsafe_allow_html=True)
        with c4:
            mv = (forecast.get("predicted_movement") or "—").upper()
            mv_color = "#10B981" if mv=="UP" else "#EF4444" if mv=="DOWN" else "#6B7280"
            st.markdown(f"**Price Direction**  \n<span style='font-size:2rem;color:{mv_color};'>{mv}</span>", unsafe_allow_html=True)
        rationale = re.sub(r'conf (\d+\.\d+)', lambda m: f"conf {fmt_pct(m.group(1))}", decision.get("rationale","—"))
        rationale = re.sub(r'thresh (\d+\.\d+)', lambda m: f"thresh {fmt_pct(m.group(1))}", rationale)
        st.info(rationale)
        st.markdown("### 🤖 LLM Explanation")
        st.write(llm_text)

with tab_forecast:
    st.subheader("📊 Forecast Details & Model Comparison")
    if not ctx:
        st.info("Run the analysis to populate this view.")
    else:
        forecast = ctx["forecast"]; mpm = forecast.get("metrics_per_model", {}); best = forecast.get("model_used","—")
        st.markdown(f"**Selected model**: **{best}**  \n**Forecast Confidence**: **{fmt_pct(forecast.get('confidence',0),1)}**")
        rows = []
        latest_price = forecast.get("latest_price", 1.0) or 1.0
        for name, met in mpm.items():
            dir_acc = to_float(met.get("Directional_Accuracy", 0))
            mape = to_float(met.get("MAPE", 0)); mae = to_float(met.get("MAE", 0))
            comp = 0.4*dir_acc + 0.3*max(0,1-mape/100) + 0.3*max(0,1-(mae/max(latest_price,1e-8)))
            rows.append({"Model":("👑 " if name==best else "")+pretty_model(name),
                         "Directional Accuracy": f"{fmt_sig(dir_acc*100,2)}%",
                         "MAPE": f"{fmt_sig(mape,2)}%", "MAE ($)": f"{fmt_sig(mae,2)}",
                         "Composite Score": f"{fmt_sig(comp*100,2)}%"})
        # If LSTM backtest exists, ensure it also appears if not in metrics
        if bt is not None and not bt.empty and include_lstm_backtest and "predicted_LSTM" in bt.columns and "LSTM" not in mpm:
            y = bt["actual_close"].values.astype(float); p = bt["predicted_LSTM"].values.astype(float)
            eps=1e-8; rmse=float(np.sqrt(((y-p)**2).mean())); mae=float(np.abs(y-p).mean()); mape=float(np.mean(np.abs((y-p)/np.maximum(np.abs(y),eps)))*100.0)
            # dir acc
            if len(y)>=2 and len(p)>=2:
                ac=np.diff(y)/y[:-1]*100; pc=np.diff(p)/p[:-1]*100; sig=np.abs(ac)>=0.1; dir_acc=0.5 if not sig.any() else float((np.sign(ac[sig])==np.sign(pc[sig])).sum()/sig.sum())
            else: dir_acc=0.5
            comp=0.4*dir_acc+0.3*max(0,1-mape/100)+0.3*max(0,1-(mae/max(latest_price,1e-8)))
            rows.append({"Model":pretty_model("LSTM"),"Directional Accuracy":f"{fmt_sig(dir_acc*100,2)}%","MAPE":f"{fmt_sig(mape,2)}%","MAE ($)":f"{fmt_sig(mae,2)}","Composite Score":f"{fmt_sig(comp*100,2)}%"})
        if rows: st.table(pd.DataFrame(rows))

        st.markdown("### 📐 Diebold–Mariano Tests (squared-error loss)")
        if dm_tbl is None or dm_tbl.empty:
            st.info("No DM results (need backtest predictions and a selected model).")
        else:
            dm_view = dm_tbl.copy()
            for col in ["DM_stat","p_value"]:
                if col in dm_view.columns:
                    dm_view[col] = dm_view[col].apply(lambda x: fmt_sig(x,2) if pd.notna(x) else "—")
            st.caption(f"Null hypothesis: no accuracy difference vs **{best}**. Significant at p < 0.10.")
            st.table(dm_view[["Model","DM_stat","p_value","Winner"]])

with tab_sentiment:
    st.subheader("🧠 Sentiment Analysis")
    if not ctx:
        st.info("Run the analysis to populate this view.")
    else:
        s = ctx["sentiment"]
        verdict=(s.get("verdict") or "neutral").upper(); strength=s.get("strength_category","Weak")
        pol=to_float(s.get("score"),0.0); conf=to_float(s.get("confidence"),0.0); vol=to_int(s.get("news_volume"),0)
        fresh=to_float(s.get("recency_score"),0.0); hist=to_float(s.get("directional_accuracy"),0.0)
        c1,c2,c3,c4=st.columns(4)
        with c1: st.metric("Verdict", verdict, strength); st.markdown(badge(verdict, verdict_color(verdict)), unsafe_allow_html=True)
        with c2: st.metric("Polarity", f"{pol:+.3f}")
        with c3: st.metric("Sentiment Confidence", fmt_pct(conf,1))
        with c4: st.metric("News Volume", f"{vol} articles")
        c5,c6=st.columns(2)
        with c5: st.metric("News Freshness", fmt_pct(fresh,1))
        with c6: st.metric("Historical Accuracy", fmt_pct(hist,1))

        if enable_finbert:
            fb = st.session_state.get("finbert", {})
            if fb:
                st.markdown("#### 🤖 FinBERT (headline-level sentiment)")
                st.table(pd.DataFrame([
                    ["Avg Positive", fmt_pct(fb.get("avg_positive", 0.0), 1)],
                    ["Avg Neutral", fmt_pct(fb.get("avg_neutral", 0.0), 1)],
                    ["Avg Negative", fmt_pct(fb.get("avg_negative", 0.0), 1)],
                    ["FinBERT Score (−1..1)", f"{fmt_sig(fb.get('avg_score', 0.0), 2)}"],
                    ["FinBERT Verdict", (fb.get("verdict","neutral") or "neutral").upper()],
                    ["FinBERT Confidence", fmt_pct(fb.get("confidence", 0.0), 1)],
                ], columns=["Metric","Value"]))
            else:
                st.info("FinBERT enabled, but no headlines available (or model not loaded). Showing proxy sentiment above.")

# ======= CORRELATION =======
with tab_corr:
    st.subheader("📈 Backtest — Actual vs Predicted")
    if bt is None or bt.empty:
        st.info("Run the analysis (and ensure enough history).")
    else:
        # View toggle: Levels / % Error / Returns
        view = st.radio("View", ["Levels", "% Error", "Returns"], horizontal=True, key="corr_view")

        # Build list of available prediction columns dynamically
        pred_cols_all = [c for c in bt.columns if c.startswith("predicted_")]
        pred_cols = [c for c in pred_cols_all if bt[c].notna().any()]

        # ----- Build plotting frame depending on view
        if view == "Levels":
            # Actual
            actual_df = bt[["date", "actual_close"]].copy()
            actual_df["Series"] = "Actual Price"
            actual_df = actual_df.rename(columns={"actual_close": "Value"})

            # Predictions (long)
            preds_long = bt[["date"] + pred_cols].melt(
                id_vars=["date"], var_name="SeriesRaw", value_name="Value"
            )
            preds_long["Series"] = preds_long["SeriesRaw"].str.replace("predicted_", "", regex=False).apply(pretty_model)
            preds_long = preds_long.drop(columns=["SeriesRaw"])

            # Combined
            plot_df = pd.concat([actual_df[["date", "Series", "Value"]], preds_long], ignore_index=True)
            y_title = "Price (USD)"

        elif view == "% Error":
            # Percent error for each model vs actual
            tmp = bt.copy()
            eps = 1e-8
            for c in pred_cols:
                tmp[c] = (tmp[c] - tmp["actual_close"]) / np.maximum(np.abs(tmp["actual_close"]), eps) * 100.0
            preds_long = tmp[["date"] + pred_cols].melt(
                id_vars=["date"], var_name="SeriesRaw", value_name="Value"
            )
            preds_long["Series"] = preds_long["SeriesRaw"].str.replace("predicted_", "", regex=False).apply(pretty_model)
            preds_long = preds_long.drop(columns=["SeriesRaw"])
            plot_df = preds_long
            y_title = "% Error"

        else:  # "Returns"
            tmp = bt.copy()
            tmp["actual_ret"] = tmp["actual_close"].astype(float).pct_change() * 100.0
            for c in pred_cols:
                tmp[c] = tmp[c].astype(float).pct_change() * 100.0

            # Actual
            actual_df = tmp[["date", "actual_ret"]].rename(columns={"actual_ret": "Value"})
            actual_df["Series"] = "Actual Price"

            # Predictions (long)
            preds_long = tmp[["date"] + pred_cols].melt(
                id_vars=["date"], var_name="SeriesRaw", value_name="Value"
            )
            preds_long["Series"] = preds_long["SeriesRaw"].str.replace("predicted_", "", regex=False).apply(pretty_model)
            preds_long = preds_long.drop(columns=["SeriesRaw"])

            # Combined
            plot_df = pd.concat([actual_df[["date", "Series", "Value"]], preds_long], ignore_index=True)
            y_title = "Return (%)"

        # ----- Encodings (distinct color + shape; single legend on color)
        plot_df["date"] = pd.to_datetime(plot_df["date"])

        present_pred_series = sorted(
            plot_df.loc[plot_df["Series"] != "Actual Price", "Series"].unique().tolist()
        )
        domain_series_all = (["Actual Price"] if view != "% Error" else []) + present_pred_series

        base_colors = {
            "Actual Price": "#222222",
            "Naive (Random Walk)": "#1f77b4",
            "Simple Exponential Smoothing (SES)": "#ff7f0e",
            "ARIMA": "#2ca02c",
            "Prophet": "#d62728",
            "LSTM": "#9467bd",
        }
        base_shapes = {
            "Actual Price": "square",
            "Naive (Random Walk)": "triangle-up",
            "Simple Exponential Smoothing (SES)": "diamond",
            "ARIMA": "circle",
            "Prophet": "cross",
            "LSTM": "triangle-down",
        }

        color_scale = alt.Scale(
            domain=domain_series_all,
            range=[base_colors.get(s, "#888888") for s in domain_series_all],
        )
        shape_scale = alt.Scale(
            domain=domain_series_all,
            range=[base_shapes.get(s, "circle") for s in domain_series_all],
        )

        x_enc = alt.X(
            "date:T",
            title="Date",
            axis=alt.Axis(format="%b %d", labelAngle=-30, labelFlush=True),
        )

        # Line layer with a single color legend
        line_layer = alt.Chart(plot_df).mark_line().encode(
            x=x_enc,
            y=alt.Y("Value:Q", title=y_title),
            color=alt.Color("Series:N", scale=color_scale, legend=alt.Legend(title="Series")),
        )

        # Point layer adds shapes (no extra legend), improves visual distinguishability
        point_layer = alt.Chart(plot_df).mark_point(size=55).encode(
            x="date:T",
            y="Value:Q",
            color=alt.Color("Series:N", scale=color_scale, legend=None),
            shape=alt.Shape("Series:N", scale=shape_scale, legend=None),
            tooltip=[
                alt.Tooltip("date:T", title="Date", format="%Y-%m-%d"),
                alt.Tooltip("Value:Q", title=y_title),
                alt.Tooltip("Series:N", title="Series"),
            ],
        )

        chart_all = (line_layer + point_layer).properties(
            height=420, title=f"All Models vs Actual (view: {view})"
        ).configure_view(stroke="#cccccc")

        st.altair_chart(chart_all, use_container_width=True)

        # --- Selected model vs Actual with identical marker rules
        sel_name = ctx["forecast"].get("model_used", "ARIMA")
        sel_pretty = pretty_model(sel_name)
        sel_col = f"predicted_{sel_name}"

        if sel_col in bt.columns:
            if view == "Levels":
                df2 = pd.DataFrame({
                    "date": pd.to_datetime(bt["date"]),
                    "Actual Price": bt["actual_close"].astype(float),
                    sel_pretty: bt[sel_col].astype(float),
                })
                y2 = "Price (USD)"
            elif view == "Returns":
                df2 = pd.DataFrame({
                    "date": pd.to_datetime(bt["date"]),
                    "Actual Price": bt["actual_close"].astype(float).pct_change() * 100.0,
                    sel_pretty: bt[sel_col].astype(float).pct_change() * 100.0,
                })
                y2 = "Return (%)"
            else:  # "% Error" -> keep a clean visual by showing level comparison for the winner
                df2 = pd.DataFrame({
                    "date": pd.to_datetime(bt["date"]),
                    "Actual Price": bt["actual_close"].astype(float),
                    sel_pretty: bt[sel_col].astype(float),
                })
                y2 = "Price (USD)"

            df2_long = df2.melt(id_vars=["date"], var_name="Series", value_name="Value")
            domain2 = ["Actual Price", sel_pretty]
            color2 = alt.Scale(domain=domain2,
                               range=[base_colors["Actual Price"], base_colors.get(sel_pretty, "#1565C0")])
            shape2 = alt.Scale(domain=domain2,
                               range=[base_shapes["Actual Price"], base_shapes.get(sel_pretty, "triangle-up")])

            line2 = alt.Chart(df2_long).mark_line().encode(
                x=x_enc,
                y=alt.Y("Value:Q", title=y2),
                color=alt.Color("Series:N", scale=color2, legend=alt.Legend(title="Series")),
            )
            pts2 = alt.Chart(df2_long).mark_point(size=65).encode(
                x="date:T",
                y="Value:Q",
                color=alt.Color("Series:N", scale=color2, legend=None),
                shape=alt.Shape("Series:N", scale=shape2, legend=None),
            )
            chart_sel = (line2 + pts2).properties(
                height=420, title=f"Selected Model vs Actual — {sel_pretty}"
            )
            st.altair_chart(chart_sel, use_container_width=True)
        else:
            st.info("Selected model series not present in backtest (insufficient data for that horizon).")


with tab_explain:
    st.subheader("🛡️ Explainability & What-if")
    if not ctx:
        st.info("Run the analysis to populate this view.")
    else:
        tr = ctx.get("explainability", {}).get("full_trace", [])
        if tr:
            for i, line in enumerate(tr, 1):
                if any(k in line for k in ["✅","completed"]): st.success(f"{i:02d}. {line}")
                elif any(k in line.lower() for k in ["❌","failed","error"]): st.error(f"{i:02d}. {line}")
                elif any(k in line for k in ["⚠️","warning"]): st.warning(f"{i:02d}. {line}")
                else: st.info(f"{i:02d}. {line}")
        else:
            st.info("No trace available.")

        st.markdown("### 🔎 What-if (instant, no retrain)")
        base_thresh = float(st.session_state.get("base_threshold", 0.70))
        fc = ctx["forecast"]; stn = ctx["sentiment"]
        move = fc.get("predicted_movement", "stable"); conf = float(fc.get("confidence", 0.0))
        cases=[("Baseline",base_thresh,stn.get("verdict","neutral")),("Lower threshold (−10pp)",max(0.0,base_thresh-0.10),stn.get("verdict","neutral")),("Higher threshold (+10pp)",min(0.99,base_thresh+0.10),stn.get("verdict","neutral")),("Force Bullish sentiment",base_thresh,"bullish"),("Force Bearish sentiment",base_thresh,"bearish")]
        rows=[]
        for label,thr,sv in cases:
            if move=="up": thr_adj = thr - 0.10 if sv=="bullish" else (thr + 0.10 if sv=="bearish" else thr)
            elif move=="down": thr_adj = thr - 0.10 if sv=="bearish" else (thr + 0.10 if sv=="bullish" else thr)
            else: thr_adj = max(0.75, thr)
            action = "BUY" if (move=="up" and conf>=thr_adj) else ("SELL" if (move=="down" and conf>=thr_adj) else "HOLD")
            rows.append({"Scenario":label,"Adjusted Threshold":f"{fmt_sig(thr_adj*100,2)}%","Model Confidence":f"{fmt_sig(conf*100,2)}%","Action":action})
        st.table(pd.DataFrame(rows))

        openai_key = os.getenv("OPENAI_API_KEY", "")
        payload2 = dict(llm_payload or {}); payload2["what_if"]=rows
        st.markdown("#### 🤖 What-if Summary")
        st.write(cached_llm_explain(openai_key, payload2))


Writing streamlit_app.py


In [ ]:
# Cell 12 — Streamlit + robust ngrok (with proper remote session cleanup)
!pip -q install streamlit pyngrok requests

import os, subprocess, time, socket, sys, psutil, getpass, json, requests
from pyngrok import ngrok

APP_FILE = "streamlit_app.py"
assert os.path.exists(APP_FILE), "❌ streamlit_app.py not found. Re-run the cell that writes it."

# ---------- Helpers ----------
def kill_like(substr: str):
    substr = substr.lower()
    for p in psutil.process_iter(attrs=["pid","name","cmdline"]):
        try:
            cmd = " ".join(p.info.get("cmdline") or []).lower()
            if substr in cmd:
                p.kill()
        except Exception:
            pass

def free_port(port: int):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if s.connect_ex(("127.0.0.1", port)) == 0:
            for p in psutil.process_iter(attrs=["pid","connections"]):
                for c in p.info.get("connections") or []:
                    if getattr(c, "laddr", None) and c.laddr.port == port:
                        p.kill()
        s.close()
    except Exception:
        pass

def wait_for_port(host="127.0.0.1", port=8501, timeout=90):
    end = time.time() + timeout
    while time.time() < end:
        try:
            s = socket.create_connection((host, port), timeout=1)
            s.close()
            return True
        except Exception:
            time.sleep(1)
    return False

def list_tunnel_sessions(api_key: str):
    """Return list of tunnel session dicts. Uses the Service API."""
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Ngrok-Version": "2",
        "Accept": "application/json",
    }
    r = requests.get("https://api.ngrok.com/tunnel_sessions", headers=headers, timeout=20)
    if r.status_code != 200:
        raise RuntimeError(f"List sessions failed ({r.status_code}): {r.text[:300]}")
    data = r.json()
    return data.get("tunnel_sessions", [])

def stop_tunnel_session(api_key: str, session_id: str):
    """POST /tunnel_sessions/{id}/stop"""
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Ngrok-Version": "2",
        "Accept": "application/json",
    }
    r = requests.post(f"https://api.ngrok.com/tunnel_sessions/{session_id}/stop",
                      headers=headers, timeout=20)
    if r.status_code not in (204,):
        # Sometimes the session dies between list and stop; ignore 404 here.
        if r.status_code != 404:
            raise RuntimeError(f"Stop failed for {session_id} ({r.status_code}): {r.text[:300]}")

# ---------- 0) Kill any local blockers ----------
kill_like("streamlit run")
kill_like("ngrok")           # kill old agent(s)
free_port(8501)              # free the Streamlit port

# ---------- 1) Start Streamlit headless ----------
env = os.environ.copy()
env["STREAMLIT_SERVER_HEADLESS"] = "true"
env["STREAMLIT_SERVER_PORT"] = "8501"
env["STREAMLIT_SERVER_ADDRESS"] = "0.0.0.0"
env["BROWSER_GATHER_USAGE_STATS"] = "false"
env["NEWS_API_KEY"] = os.environ.get("NEWS_API_KEY","")
env["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY","")

log_path = "streamlit.log"
log = open(log_path, "w")
sp = subprocess.Popen(["streamlit", "run", APP_FILE], stdout=log, stderr=log, env=env)

if not wait_for_port():
    log.close()
    print("❌ Streamlit did not start on :8501.\n---- Last 200 log lines ----")
    try:
        print("".join(open(log_path).readlines()[-200:]))
    except Exception:
        print("(no log output)")
    raise SystemExit

print("✅ Streamlit is up at http://127.0.0.1:8501")

# ---------- 2) (Optional) Clean up remote agent sessions via Service API ----------
print("🔑 Enter your ngrok API KEY (Personal Access Token) for remote cleanup (press Enter to skip): ", end="")
api_key = getpass.getpass("")

if api_key.strip():
    try:
        sessions = list_tunnel_sessions(api_key.strip())
        if not sessions:
            print("ℹ️ No remote tunnel sessions found.")
        else:
            print(f"🧹 Found {len(sessions)} session(s). Stopping…")
            stopped = 0
            for s in sessions:
                sid = s.get("id")
                try:
                    stop_tunnel_session(api_key.strip(), sid)
                    stopped += 1
                    print("   • Stopped:", sid)
                except Exception as e:
                    print("   • Couldn’t stop", sid, "-", e)
            print(f"✅ Remote cleanup complete. Stopped {stopped} session(s).")
    except Exception as e:
        print(f"⚠️ Remote cleanup skipped (API error): {e}")

# ---------- 3) Start a fresh pyngrok tunnel with your AUTHTOKEN ----------
print("🔑 Enter your ngrok AUTHTOKEN (for the agent): ", end="")
authtoken = getpass.getpass("")

if authtoken.strip():
    try:
        ngrok.set_auth_token(authtoken.strip())
    except Exception:
        # Older pyngrok versions also respect NGROK_AUTHTOKEN env var
        os.environ["NGROK_AUTHTOKEN"] = authtoken.strip()

# Ensure any old pyngrok child process is gone
kill_like("pyngrok")

# Try connect (free tier → 1 agent session allowed)
try:
    print("⏳ Starting ngrok tunnel…")
    try:
        tun = ngrok.connect(addr=8501, proto="http", bind_tls=True)
    except TypeError:
        tun = ngrok.connect(addr=8501, proto="http")
    print("✅ Public URL:", tun.public_url)
    print("⚙️  Local URL:  http://127.0.0.1:8501")
    print(f"🧾 Logs: tail -n 200 {log_path}")
except Exception as e:
    msg = str(e)
    print("\n❌ ngrok failed to start.\n")
    print(msg)
    print("""
➡ Another ngrok agent is still connected for this account OR the authtoken is invalid.
   - Make sure you used your **API key** above for cleanup and your **authtoken** here for the agent.
   - You can also terminate sessions in the dashboard:
     https://dashboard.ngrok.com/agents
   - As a last resort, rotate your authtoken (kills old agents):
     https://dashboard.ngrok.com/get-started/your-authtoken
""")
    raise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 64.3 MB/s eta 0:00:00
✅ Streamlit is up at http://127.0.0.1:8501
🔑 Enter your ngrok API KEY (Personal Access Token) for remote cleanup (press Enter to skip): ··········
⚠️ Remote cleanup skipped (API error): List sessions failed (400): {"error_code":"ERR_NGROK_206","status_code":400,"msg":"The authentication you specified is actually an authtoken. Your credential: '2tQZRBJNcmBlNmxw14ypKMm2I01_73o4TmSLAuX9ebo1xXtdV'. Check your records for an API key. API keys and instructions are available on your dashboard: https://dashboard.ngro
🔑 Enter your ngrok AUTHTOKEN (for the agent): ··········
⏳ Starting ngrok tunnel…
✅ Public URL: https://22d7f606426e.ngrok-free.app
⚙️  Local URL:  http://127.0.0.1:8501
🧾 Logs: tail -n 200 streamlit.log


In [ ]:
##2tQZRBJNcmBlNmxw14ypKMm2I01_73o4TmSLAuX9ebo1xXtdV